# Тюмень: исправленная квартальная структура (refined)

Это отдельный ноутбук для повторного исследования Тюмени с исправленной
квартальной структурой. Он **не читает и не перезаписывает** результаты
первоначальной версии.

- Первоначальная версия: `tyumen_pipeline_original.ipynb` → `examples/data_tyumen/original/`
- Исправленная версия: этот ноутбук → `examples/data_tyumen/refined/`

> Ранее сделанный `examples/data_tyumen_pbf/full_fixed/` — промежуточный
> исправленный прогон без дополнительного split крупных кварталов. Он сохранён
> отдельно; этот ноутбук его не читает и не изменяет.

> **Совместимость входов проверена:** PBF и `graph.pkl` совпадают с исходным
> прогоном, `functional_zones.geojson` — тот же файл, границы геометрически
> эквивалентны. Слои `buildings_raw` и `schools_raw` получены из того же PBF.
> Кварталы, матрица и результаты `full_fixed` не используются.

Ноутбук использует только локальный PBF `tyumen_refined.osm.pbf`. Overpass API
и скачивание данных не используются.

## Что пересчитывается

1. Дороги читаются с `filter_type='keep'` и с `highway=service`.
2. Здания загружаются до нарезки и используются для дополнительного разделения
   крупных кварталов.
3. Кварталы нарезаются заново; крупные кварталы с застройкой дополнительно
   делятся по зданиям и линиям (порог задаётся в настройках).
4. Землепользование, граф, матрица доступности, здания, население, школы,
   обеспеченность, плотность, морфотипы и доступность территории считаются
   заново для новой структуры.
5. В конце есть проверки качества и сравнение с `original`.

> Ноутбук не запускает расчёты при открытии. Выполняйте ячейки сверху вниз или
> `Kernel → Restart Kernel and Run All Cells`.


## Перед запуском: окружение и зависимости

**Ядро (kernel) ноутбука**: нужно выбрать интерпретатор из папки `.venv-blocksnet` (Python 3.12).

Jupyter и ipykernel уже установлены в это окружение. Если понадобится переустановить:

```
.venv-blocksnet/Scripts/python.exe -m pip install jupyterlab ipykernel
```

Запуск Jupyter из корня проекта:

```
.venv-blocksnet/Scripts/python.exe -m jupyter lab
```

Затем откройте файл `examples/tyumen_pipeline_refined.ipynb` и выберите ядро `.venv-blocksnet`.

**Зависимости проекта** (`pyproject.toml`). Данные OpenStreetMap читаются из локального архива (`.osm.pbf`) библиотекой **pyrosm**; в `dependencies` уже добавлена:

```
"pyrosm>=0.23",
```

`blocksnet`, `osmnx`, `geopandas`, `pyarrow` и остальные пакеты установлены.

> **Данные не скачиваются.** В `examples/data_tyumen/refined/` уже лежит локальный PBF. Ноутбук только проверяет его наличие. Overpass API не используется.

> **Требования к памяти.** Для расчёта матрицы и показателей нужно ~1–2 ГБ свободной ОЗУ. Ноутбук автоматически уменьшит порцию матрицы, если свободной памяти меньше 2 ГБ.


## Кратко о терминах

- **Квартал** — небольшой участок города, ограниченный улицами, рекой или железной дорогой.
- **Граф** — сеть дорог и дорожек, по которой считаются маршруты. Здесь используется готовый **пешеходный** граф OpenStreetMap.
- **Матрица доступности** — таблица «сколько минут идти из одного квартала в другой». Размер N×N, где N — число кварталов.
- **Показатели** — числа, которые описывают кварталы (население, обеспеченность школами, плотность застройки и т.д.).

> Интермодальный граф (с общественным транспортом) не строим — по условию задания.


## Справочник метрик BlocksNet

BlocksNet — это конструктор: единого списка «метрик» нет, есть набор функций, каждая возвращает таблицу с показателями. Ниже — каталог по группам. Единицы указаны в скобках; если единицы нет — это доля или коэффициент.

### 1. Пространственные отношения
- **Матрица доступности** (`calculate_accessibility_matrix`) — время в пути между центрами кварталов, **минуты**. Размер N×N.
- **Матрица расстояний** (`calculate_distance_matrix`) — расстояние по прямой между центрами кварталов, **метры**.
- **Граф смежности** (`generate_adjacency_graph`) — какие кварталы граничат друг с другом. Без единиц.
- **OD-матрица** (`origin_destination_matrix`) — условные потоки «откуда—куда» с учётом притягательности кварталов. Относительные единицы.
- **Загруженность дорог** (`road_congestion`) — суммарный поток на участок дороги. Условные единицы.

### 2. Геометрия кварталов
- `site_area` — площадь квартала, **м²**.
- `site_length` — периметр квартала, **м**.
- `outer_radius` / `inner_radius` — внешний и внутренний радиусы квартала, **м**.
- `centerline_length` — длина осевой линии квартала, **м**.
- `aspect_ratio` — вытянутость квартала (отношение длинной стороны к короткой), **без единиц**, ≥ 1.

### 3. Плотность и застройка
- `footprint_area` — площадь застройки (пятно зданий), **м²**.
- `build_floor_area` — суммарная поэтажная площадь, **м²**.
- `living_area` / `non_living_area` — жилая и нежилая площадь, **м²**.
- `fsi` — коэффициент плотности застройки (build_floor_area / site_area), **м²/м²**.
- `gsi` — коэффициент застройки (footprint_area / site_area), **0…1**.
- `mxi` — доля жилой площади (living_area / build_floor_area), **0…1**.
- `l` — средняя этажность (build_floor_area / footprint_area), **этажи**.
- `osr` — обеспеченность открытыми пространствами ((site_area − footprint_area) / build_floor_area), **м²/м²**.
- `share_living` / `share_non_living` — жилая и нежилая площадь на единицу пятна застройки, **м²/м²**.

### 4. Землепользование
- `calculate_land_use_shares` — доли типов землепользования по площади, **0…1**.
- `land_use_collocation` — совместимость землепользования соседних кварталов, **0…1**.
- `land_use_similarity` — сходство набора услуг квартала с эталонными профилями + предполагаемый тип, **0…1**.
- `land_use_prediction` — предсказание типа землепользования (машинное обучение), метка + **вероятность 0…1**.
- `calculate_land_use_indicators` — оценка застройки и населения по нормативам, когда нет данных по зданиям (**м²**, **чел.**).

### 5. Услуги
- `services_count` / `count_<тип>` — число объектов услуг, **шт.**
- `services_density` — число объектов на площадь, в коде **объектов/м²** (удобнее пересчитывать в объектов/га или /км²).
- `services_collocation` — совместная встречаемость услуг (IoU), **0…1**.
- `shannon_diversity` — разнообразие услуг, **0…ln N** (натуральные единицы).

### 6. Обеспеченность услугами
- `shared_provision` — «простая» обеспеченность: `capacity_within` (мест/объектов в радиусе), `population_within` (**чел.**), `load` (нагрузка, **доля**), `provision` (ёмкость / население, **доля**).
- `competitive_provision` — конкуренция за услуги: `demand` и `capacity` (**мест**), `demand_within` / `demand_without` / `demand_left` (**мест**), `capacity_within` / `capacity_without` / `capacity_left` (**мест**), `provision_strong` / `provision_weak` (**0…1**).
- `links_df` — какие кварталы и сколько мест передали друг другу (пары `from`, `to`, `value`).
- `provision_strong_total` / `provision_weak_total` — те же доли, но сразу по всей территории, **0…1**.

### 7. Доступность и сеть
- `mean_accessibility` / `median_accessibility` / `max_accessibility` — среднее, медианное и максимальное время до всех кварталов, **минуты**.
- `area_accessibility` — средневзвешенное по площади время до всех кварталов, **минуты**.
- `relative_accessibility` — время относительно самого доступного квартала, **минуты**.
- `calculate_connectivity` — связность, **1/минуты** (чем меньше время, тем выше).
- `land_use_accessibility` / `land_use_accessibility_matrix` — время до кварталов заданного типа, **минуты**.
- `NetworkClassifier` — тип уличной сети (метка) и вероятности классов, **0…1**.

### 8. Морфотипы застройки
- `get_spacematrix_morphotypes` — кластеры и морфотипы по трём показателям: `fsi` (**м²/м²**), `mxi` (**0…1**), `l` (**этажи**); плюс интерпретации.
- `get_strelka_morphotypes` — морфотипы Strelka по тем же показателям: низкая, средняя, высокая застройка и их подтипы.

### 9. Центральность
- `population_centrality` — «центральность» квартала по населению и связности, **0…1**.
- `services_centrality` — концентрация услуг с учётом связности, разнообразия и плотности, **0…1**.
- `degree_centrality` — доля возможных соседей у квартала, **0…1**.

### 10. Социально-экономические показатели
Набор `SocioEconomicAggregator` считает показатели по группам:
- **общие** — площадь (**км²**), уровень урбанизации (**0…1**);
- **демография** — население (**чел.**), плотность (**чел./км²**), рождаемость, смертность, миграция (**на человека**);
- **инженерия** — количество объектов инфраструктуры (**шт.**);
- **транспорт** — плотность дорог (**км/км²**), длина дорог (**км**), связность, число АЗС/вокзалов/аэропортов (**шт.**), средняя доступность (**мин**);
- **социальная сфера** — количество объектов по типам (**шт.**) и обеспеченность (**на человека**).

### 11. Пустующие территории
- `get_vacant_areas` — свободные от застройки участки: площадь (**м²**), площадь минимального прямоугольника (**м²**), периметр (**м**), отношения (**без единиц**), номер квартала.

> В нашем ноутбуке используется только часть этих метрик — те, что есть в `pipeline.ipynb`. Остальные можно добавить позже, если они нужны для выводов.


## Показатели из `pipeline.ipynb`

Ниже — все показатели, которые считает оригинальный пример `pipeline.ipynb`, и то, что они дадут по Тюмени. В нашем ноутбуке набор тот же; отличие только в графе: в примере интермодальный, у нас пешеходный (`walk`), поэтому время в пути будет больше.

### Матрица доступности
| Показатель | Что означает | Единицы | Что расскажет о Тюмени |
|---|---|---|---|
| `acc_mx` | Время в пути между центрами каждой пары кварталов | минуты | Какие районы связаны хорошо, а какие изолированы; сколько минут идти до школы, центра, соседнего квартала |

### Здания и население (агрегация по кварталам)
| Показатель | Что означает | Единицы | Что расскажет о Тюмени |
|---|---|---|---|
| `population` | Суммарное население квартала | чел. | Где живёт больше людей; нагрузка на школы и инфраструктуру |
| `footprint_area` | Площадь застройки (пятно зданий) | м² | Насколько плотно занята территория |
| `build_floor_area` | Суммарная поэтажная площадь | м² | Общий объём застройки |
| `living_area` | Жилая площадь | м² | Жилищный фонд района |
| `non_living_area` | Нежилая площадь | м² | Офисы, магазины, производство |
| `objects_count` | Число зданий в квартале | шт. | Размер и дробность застройки |

### Школы
| Показатель | Что означает | Единицы | Что расскажет о Тюмени |
|---|---|---|---|
| `capacity` | Суммарная ёмкость школ квартала | мест | Где есть запас мест, а где его нет |
| `objects_count` | Число школ в квартале | шт. | Насыщенность района школами |

### Землепользование (`assign_land_use`)
| Показатель | Что означает | Единицы | Что расскажет о Тюмени |
|---|---|---|---|
| `residential`, `business`, `recreation`, `industrial`, `transport`, `special`, `agriculture` | Доли типов землепользования внутри квартала | 0…1 | Из чего состоит квартал: жильё, бизнес, промзона, рекреация |
| `land_use` | Преобладающий тип землепользования квартала | метка | Главная функция квартала |
| `share` | Доля преобладающего типа | 0…1 | Насколько квартал однороден |

### Обеспеченность школами (`competitive_provision`)
| Показатель | Что означает | Единицы | Что расскажет о Тюмени |
|---|---|---|---|
| `demand` | Нормативная потребность в местах (население / 1000 × 120) | мест | Сколько мест нужно району по нормативу |
| `capacity` | Ёмкость школ | мест | Сколько мест есть |
| `demand_within` | Потребность, покрытая в пределах норматива доступности | мест | Сколько детей могут дойти до школы за 15 минут |
| `demand_without` | Потребность, покрытая только за пределами норматива | мест | Сколько детей учатся далеко от дома |
| `demand_left` | Непокрытая потребность | мест | Дефицит мест |
| `capacity_within` / `capacity_without` / `capacity_left` | Ёмкость, использованная в пределах норматива / за его пределами / оставшаяся | мест | Где школы перегружены, а где есть свободные места |
| `provision_strong` | Доля потребности, покрытая в пределах 15 минут | 0…1 | Главный показатель: 1 — все обеспечены рядом, 0 — рядом мест нет |
| `provision_weak` | Доля потребности с учётом поездок за пределы норматива | 0…1 | Насколько ситуацию спасают дальние школы |
| `links_df` | Связки «квартал-потребитель → квартал-школа» и число переданных мест | пары, мест | Кто на самом деле «кормит» школу и откуда идут дети |

### Плотность и застройка (`calculate_density_indicators`)
| Показатель | Что означает | Единицы | Что расскажет о Тюмени |
|---|---|---|---|
| `site_area` | Площадь квартала | м² | Размер кварталов |
| `footprint_area` | Площадь застройки | м² | Сколько земли занято зданиями |
| `build_floor_area` | Поэтажная площадь | м² | Объём застройки |
| `living_area` / `non_living_area` | Жилая / нежилая площадь | м² | Баланс жилья и других функций |
| `fsi` | Плотность застройки | м²/м² | Насколько интенсивно используется земля |
| `gsi` | Доля застроенной земли | 0…1 | Насколько «заполнен» квартал зданиями |
| `mxi` | Доля жилья в застройке | 0…1 | Жилой или нежилой квартал |
| `l` | Средняя этажность | этажи | Низкая, средняя или высокая застройка |
| `osr` | Обеспеченность открытыми пространствами | м²/м² | Сколько свободной земли приходится на застройку |
| `share_living` / `share_non_living` | Жилая / нежилая площадь на единицу пятна | м²/м² | Компактность и функция застройки |

### Морфотипы
| Показатель | Что означает | Единицы | Что расскажет о Тюмени |
|---|---|---|---|
| `spacematrix_df` | Для каждого квартала: `l`, `fsi`, `mxi`, номер кластера, морфотип | этажи, м²/м², 0…1, метка | Типы городской среды: исторический центр, советская застройка, современные микрорайоны, промзоны |
| `spacematrix_clusters` | Средние `l`, `fsi`, `mxi` по кластеру и их интерпретации | этажи, м²/м², 0…1 | Портрет каждого типа застройки |
| `strelka_df` | Морфотип Strelka для каждого квартала | метка | Более привычные названия типов среды |

### Доступность территории
| Показатель | Что означает | Единицы | Что расскажет о Тюмени |
|---|---|---|---|
| `area_accessibility` | Средневзвешенное по площади время до всех кварталов | минуты | Насколько город в целом компактен и связен; где окраины «отрезаны» |

### Что важно помнить
- Норматив по школам в BlocksNet: **120 мест на 1000 жителей**, нормативная доступность — **15 минут**.
- Обеспеченность считается с конкуренцией: вместимость ближайших школ делится между соседними кварталами, а не достаётся каждому целиком.
- В нашем ноутбуке граф **пешеходный**, поэтому время в пути больше, чем в оригинальном примере с интермодальным графом. Это нормально и влияет на `provision_strong` и `area_accessibility`.
- Данные OSM неполные: этажность и ёмкость школ могут быть восстановлены по нормативам. Это оценочные значения, а не официальная статистика.
- `pipeline.ipynb` не считает доли землепользования (`calculate_land_use_shares`), разнообразие услуг и центральность — эти метрики есть в BlocksNet, но не входят в обязательный набор. Их можно добавить отдельно.


## Соответствие требованиям второй практики

| Требование | Где в refined-ноутбуке |
|---|---|
| Территория — город в административных границах | раздел 1, OSM relation `R2049854` |
| OpenStreetMap без Overpass | раздел 2, локальный PBF |
| Дороги, железные дороги, вода, землепользование, здания, школы | разделы 2–4, 6, 8–9 |
| Квартальная структура | раздел 4 |
| Дополнительное разделение крупных кварталов | раздел 5 |
| Пешеходный граф | раздел 7 |
| Матрица доступности N×N порциями | раздел 7 |
| Здания и население | разделы 3, 8 |
| Школы, дедупликация и обеспеченность | разделы 9–10 |
| Плотность, Spacematrix, Strelka | раздел 10 |
| Доступность территории | раздел 10 |
| Проверки качества | раздел 11 |
| Сравнение «до/после» | раздел 12 |

Все показатели из `pipeline.ipynb` сохранены: матрица доступности, население и
параметры зданий, школы и ёмкость, землепользование, обеспеченность школами,
плотностные индикаторы, морфотипы Spacematrix и Strelka, доступность территории
и сводная таблица.


In [ ]:
# Импорты и настройки (refined)
import gc
import hashlib
import json
import math
import os
import pickle
import shutil
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
from shapely.geometry import Point, box

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

# ===== НАСТРОЙКИ ИССЛЕДОВАНИЯ =====
# Это отдельная папка исправленной версии. Не меняйте её на original.
MODE = 'refined'

GRAPH_TYPE = 'walk'
CITY_NAME = 'Тюмень, Тюменская область, Россия'
CITY_OSM_ID = 'R2049854'
CITY_CENTER = (65.5412, 57.1522)
CITY_AREA_RANGE_KM2 = (450.0, 530.0)

# Данные не скачиваются: используется локальный PBF.
OSM_ARCHIVE_BUFFER_KM = 3.0
OSM_SOURCE_NAME = 'Ural Federal District (Geofabrik), local PBF'
OSM_ARCHIVE_URL = 'https://download.geofabrik.de/russia/ural-fed-district-latest.osm.pbf'
OSM_ARCHIVE_MD5_URL = OSM_ARCHIVE_URL + '.md5'
OSM_ARCHIVE_SIZE_BYTES = 396_075_670
OSM_ARCHIVE_LAST_MODIFIED = '2026-10-18'
OSM_ARCHIVE_MD5 = 'a190b259d687acc3f966c3ff19fa735e'
OSM_VERIFY_MD5 = False
OSM_USER_AGENT = 'tyumen-blocksnet-refined/1.0 (educational project; pyrosm)'

# Запуск из examples/ или из корня репозитория.
BASE_DIR = Path.cwd()
if not (BASE_DIR / 'data_tyumen').exists() and (BASE_DIR / 'examples' / 'data_tyumen').exists():
    BASE_DIR = BASE_DIR / 'examples'
DATA_ROOT = BASE_DIR / 'data_tyumen'
DATA_DIR = DATA_ROOT / MODE
DATA_DIR.mkdir(parents=True, exist_ok=True)
OSM_RAW_PATH = DATA_ROOT / 'osm_raw' / 'ural-fed-district-latest.osm.pbf'
OSM_PBF_PATH = DATA_DIR / f'tyumen_{MODE}.osm.pbf'

# ===== КВАРТАЛЫ =====
MIN_BLOCK_AREA_M2 = 500.0

# Дополнительное разделение крупных кварталов.
# Порог 1 км²: кварталы больше этого порога с застройкой пытаемся разделить.
# Пустые крупные полигоны (без зданий) не дробим искусственно.
SPLIT_LARGE_BLOCKS = True
SPLIT_MAX_AREA_KM2 = 1.0
SPLIT_MAX_AREA_M2 = SPLIT_MAX_AREA_KM2 * 1_000_000
USE_ML_CLASSIFIER = False   # True требует BlocksClassifier.default() и pygeoops

# ===== НАСТРОЙКИ РАСЧЁТА МАТРИЦЫ ДОСТУПНОСТИ =====
MATRIX_CHUNK_SIZE = 50
MATRIX_WORKERS = min(2, os.cpu_count() or 1)
SAVE_MATRIX_CSV = True
SAVE_GRAPH_LAYERS = False

try:
    import psutil
    _available_gb = psutil.virtual_memory().available / 1e9
except Exception:
    psutil = None
    _available_gb = None

if _available_gb is not None and _available_gb < 2.0:
    MATRIX_CHUNK_SIZE = 50
    MATRIX_WORKERS = 1
    print('Свободной ОЗУ меньше 2 ГБ: автоматически уменьшаем порцию матрицы.')

if DATA_DIR.name != 'refined':
    raise RuntimeError(f'Ожидалась папка refined, получена: {DATA_DIR}')
if 'original' in str(DATA_DIR).lower():
    raise RuntimeError('Refined-ноутбук не должен писать в папку original.')

print('MODE =', MODE, '| версия: исправленная квартальная структура')
print('GRAPH_TYPE =', GRAPH_TYPE)
print('Папка данных:', DATA_DIR.resolve())
print('Локальный архив OSM:', OSM_PBF_PATH.resolve())
print('Дополнительное разделение кварталов:', SPLIT_LARGE_BLOCKS,
      '| порог, км²:', SPLIT_MAX_AREA_KM2)
print('Порция матрицы:', MATRIX_CHUNK_SIZE, 'строк | потоков:', MATRIX_WORKERS)
print('Загрузчик данных: локальный PBF. Overpass API и скачивание НЕ используются.')


In [ ]:
# Вспомогательные функции: сохраняем данные в GeoJSON/Parquet/CSV и переиспользуем их.
# Важно: GeoJSON не хранит индекс таблицы, поэтому индекс сохраняется отдельным столбцом id.

def save_gdf(gdf, name, index_col='id'):
    # Пишем сначала во временный файл, потом переименовываем.
    # Так сбой посреди записи не оставит «обрезанный» GeoJSON.
    path = DATA_DIR / f'{name}.geojson'
    tmp_path = path.with_name(path.name + '.tmp')
    out = gdf.copy().reset_index()
    if 'index' in out.columns:
        out = out.rename(columns={'index': index_col})
    out.to_file(tmp_path, driver='GeoJSON')
    tmp_path.replace(path)
    print(f'Сохранено: {path.name}  (объектов: {len(out)})')


def load_gdf(name, index_col='id'):
    path = DATA_DIR / f'{name}.geojson'
    gdf = gpd.read_file(path)
    if index_col in gdf.columns:
        gdf = gdf.set_index(index_col)
    print(f'Прочитано из файла: {path.name}  (объектов: {len(gdf)})')
    return gdf


def gdf_exists(name):
    return (DATA_DIR / f'{name}.geojson').exists()


def save_df(df, name, **kwargs):
    # Пишем сначала во временный файл, потом переименовываем.
    # Так сбой посреди записи не оставит «обрезанный» CSV.
    path = DATA_DIR / f'{name}.csv'
    tmp_path = path.with_name(path.name + '.tmp')
    df.to_csv(tmp_path, **kwargs)
    tmp_path.replace(path)
    print(f'Сохранено: {path.name}')


def df_exists(name):
    return (DATA_DIR / f'{name}.csv').exists()


def load_df(name, index_col=0, **kwargs):
    path = DATA_DIR / f'{name}.csv'
    df = pd.read_csv(path, index_col=index_col, **kwargs)
    print(f'Прочитано из файла: {path.name}')
    return df


def save_parquet(df, name):
    path = DATA_DIR / f'{name}.parquet'
    tmp_path = path.with_name(path.name + '.tmp')
    df.to_parquet(tmp_path)
    tmp_path.replace(path)
    print(f'Сохранено: {path.name}  ({df.shape[0]} x {df.shape[1]})')


def parquet_exists(name):
    return (DATA_DIR / f'{name}.parquet').exists()


def load_parquet(name):
    path = DATA_DIR / f'{name}.parquet'
    df = pd.read_parquet(path)
    print(f'Прочитано из файла: {path.name}  ({df.shape[0]} x {df.shape[1]})')
    return df


def save_json(obj, name):
    path = DATA_DIR / f'{name}.json'
    tmp_path = path.with_name(path.name + '.tmp')
    tmp_path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp_path.replace(path)
    print(f'Сохранено: {path.name}')


def load_json(name):
    path = DATA_DIR / f'{name}.json'
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))


def save_graph(graph, name='graph'):
    path = DATA_DIR / f'{name}.pkl'
    tmp_path = path.with_name(path.name + '.tmp')
    with open(tmp_path, 'wb') as f:
        pickle.dump(graph, f)
    tmp_path.replace(path)
    print(f'Сохранён граф: {path.name}')


def graph_exists(name='graph'):
    return (DATA_DIR / f'{name}.pkl').exists()


def load_graph(name='graph'):
    path = DATA_DIR / f'{name}.pkl'
    with open(path, 'rb') as f:
        graph = pickle.load(f)
    print(f'Прочитан граф: {path.name}')
    return graph


def release_osm_cache():
    """Освобождает разобранный pyrosm-архив (если он больше не нужен)."""
    global _osm
    if '_osm' in globals() and _osm is not None:
        _osm = None
        gc.collect()
        print('Кэш pyrosm освобождён.')


def memory_report(tag=''):
    if psutil is None:
        return
    vm = psutil.virtual_memory()
    print(f'[память{(" " + tag) if tag else ""}] доступно {vm.available/1e9:.2f} ГБ '
          f'из {vm.total/1e9:.2f} ГБ')


print('Функции сохранения и загрузки готовы.')

## 1. Границы территории

Берём административные границы **города Тюмени** в Тюменской области — именно города (городского округа), а не всей области и не района. Границы запрашиваются из OpenStreetMap по закреплённому OSM relation `R2049854` (https://www.openstreetmap.org/relation/2049854), затем проверяются:

- геометрия — полигон или мультиполигон, валидный и непустой;
- площадь — 450–530 км² (фактически около 488,7 км²);
- координаты — в разумном диапазоне вокруг Тюмени;
- центр города лежит внутри границы.

В режиме `test` вместо всего города используется маленький участок вокруг центра Тюмени — чтобы быстро проверить, что весь расчёт работает.

> Если геокодер временно недоступен, ноутбук может взять ранее проверенные границы из `examples/data_tyumen/full/boundaries.geojson` (они относятся к тому же OSM relation) и всё равно выполнит проверки площади и геометрии.


In [ ]:
# Границы территории
def _validate_city_boundaries(gdf):
    """Проверяет, что геокодер вернул именно город Тюмень, и чистит геометрию."""
    if gdf is None or len(gdf) == 0:
        raise RuntimeError('Геокодер не вернул ни одного объекта.')

    info_cols = [c for c in ['name', 'display_name', 'type', 'addresstype', 'osm_type', 'osm_id']
                 if c in gdf.columns]
    if info_cols:
        print('Геокодер вернул объект:')
        display(gdf[info_cols].head())

    text = ' '.join(str(gdf[c].iloc[0]) for c in ['name', 'display_name'] if c in gdf.columns).lower()
    if 'тюмен' not in text:
        raise RuntimeError('Геокодер вернул не Тюмень. Проверьте CITY_NAME и CITY_OSM_ID.')
    if 'район' in text and 'городской округ' not in text:
        raise RuntimeError('Геокодер вернул район, а нужен весь город Тюмень. '
                           'Уточните CITY_NAME или CITY_OSM_ID.')

    gdf = gdf[gdf.geom_type.isin(['Polygon', 'MultiPolygon'])].copy()
    if len(gdf) == 0:
        raise RuntimeError('Геокодер не вернул полигон границы города Тюмени.')

    # Исправляем возможные самопересечения, затем собираем мультиполигон.
    bad = ~gdf.geometry.is_valid
    if bad.any():
        gdf.loc[bad, 'geometry'] = gdf.loc[bad, 'geometry'].make_valid()
    gdf = gdf[gdf.geom_type.isin(['Polygon', 'MultiPolygon'])].copy()
    gdf = gdf.dissolve().reset_index(drop=True)

    if gdf.geometry.is_empty.any() or not gdf.geometry.is_valid.all():
        raise RuntimeError('Полученная граница пустая или невалидная.')

    minx, miny, maxx, maxy = gdf.total_bounds
    if not (60.0 < minx < 70.0 and 60.0 < maxx < 70.0 and 55.0 < miny < 60.0 and 55.0 < maxy < 60.0):
        raise RuntimeError(f'Границы вне ожидаемого района Тюмени: {gdf.total_bounds}')

    area_km2 = gdf.to_crs(gdf.estimate_utm_crs()).area.sum() / 1e6
    print('Площадь границы, км2:', round(area_km2, 2))
    lo, hi = CITY_AREA_RANGE_KM2
    if not (lo <= area_km2 <= hi):
        raise RuntimeError(
            f'Площадь границы {area_km2:.1f} км2 не похожа на город Тюмень '
            f'(ожидалось {lo:.0f}–{hi:.0f} км2). Проверьте CITY_OSM_ID.'
        )

    center = Point(*CITY_CENTER)
    if not gdf.contains(center).any():
        raise RuntimeError('Центр Тюмени не попал внутрь полученной границы.')

    print('Проверка границы пройдена: город Тюмень,', round(area_km2, 1), 'км2')
    return gdf[['geometry']].reset_index(drop=True)


def _city_boundaries_from_osm():
    """Административные границы города из OpenStreetMap (Nominatim).

    Возвращает полигон именно города Тюмени, а не района или области.
    """
    try:
        if CITY_OSM_ID:
            gdf = ox.geocode_to_gdf(CITY_OSM_ID, by_osmid=True)
        else:
            gdf = ox.geocode_to_gdf(CITY_NAME)
        return _validate_city_boundaries(gdf)
    except Exception as exc:
        legacy = Path('data_tyumen/full/boundaries.geojson')
        if legacy.exists():
            print('Не удалось получить границы через геокодер:', exc)
            print('Используем ранее проверенные границы из', legacy, '(тот же OSM relation).')
            gdf = gpd.read_file(legacy)
            return _validate_city_boundaries(gdf)
        raise RuntimeError(
            'Не удалось получить границы Тюмени через OSM Nominatim и нет локального файла '
            f'{legacy}. Проверьте интернет или задайте CITY_OSM_ID.'
        ) from exc


def get_boundaries():
    # Границы уже подготовлены и лежат в папке версии. Ноутбук не скачивает
    # их и не использует файлы другой версии.
    if gdf_exists('boundaries'):
        return load_gdf('boundaries')

    raise RuntimeError(
        'Не найден boundaries.geojson в папке этой версии. '
        'Скопируйте подготовленный файл границ; скачивание в ноутбуке не выполняется.'
    )


boundaries = get_boundaries()
boundaries.plot(figsize=(8, 8), color='#cfe3ff', edgecolor='black')
plt.title('Территория исследования: Тюмень')
plt.axis('off')
plt.show()

utm = boundaries.estimate_utm_crs()
print('Проекция для расчётов:', utm)
print('Площадь территории, км2:', round(boundaries.to_crs(utm).area.sum() / 1e6, 2))


## 2. Данные OpenStreetMap: локальный архив

Overpass API не используется, интернет не нужен. В папке
`data_tyumen/refined/` уже лежит локальный архив `tyumen_refined.osm.pbf`.
Ноутбук только проверяет его наличие и размер.

Если файла нет, выполнение остановится с ошибкой. Это защищает от случайной
подмены исходных данных. Скачивание в этом ноутбуке не выполняется.


In [ ]:
# Проверка локального архива OSM. Скачивания нет.
if 'boundaries' not in globals():
    raise RuntimeError('Сначала выполните ячейку с границами территории.')

if not OSM_PBF_PATH.exists():
    raise RuntimeError(
        f'Не найден локальный архив OSM: {OSM_PBF_PATH}. '
        'Скопируйте локальный PBF в папку refined.'
    )
if OSM_PBF_PATH.stat().st_size < 100_000:
    raise RuntimeError(f'Локальный архив слишком мал: {OSM_PBF_PATH}')

minx, miny, maxx, maxy = boundaries.to_crs(4326).total_bounds
lat_mid = (miny + maxy) / 2
dlat = OSM_ARCHIVE_BUFFER_KM / 111.0
dlon = OSM_ARCHIVE_BUFFER_KM / (111.0 * math.cos(math.radians(lat_mid)))
bbox = [minx - dlon, miny - dlat, maxx + dlon, maxy + dlat]

if not (DATA_DIR / 'osm_source.json').exists():
    save_json({
        'source': OSM_SOURCE_NAME,
        'url': OSM_ARCHIVE_URL,
        'license': 'ODbL 1.0',
        'note': 'local PBF; no download in this notebook',
        'buffer_km': OSM_ARCHIVE_BUFFER_KM,
        'bbox_4326': bbox,
        'cropped_pbf': OSM_PBF_PATH.name,
        'cropped_size_bytes': OSM_PBF_PATH.stat().st_size,
        'boundary_osm_id': CITY_OSM_ID,
    }, 'osm_source')

print('Локальный архив OSM:', OSM_PBF_PATH)
print('Размер, МБ:', round(OSM_PBF_PATH.stat().st_size / 1e6, 2))
print('Overpass API НЕ используется; скачивание НЕ выполняется.')


### Дороги, железные дороги, вода

Эти объекты нужны, чтобы «разрезать» город на кварталы. Они читаются из локального архива OSM (раздел выше) и сохраняются в GeoJSON, поэтому при повторном запуске не пересчитываются.


In [ ]:
# Какие объекты OpenStreetMap берём для нарезки кварталов.
# ВАЖНО: для get_network(custom_filter=dict) pyrosm по умолчанию использует
# filter_type='exclude'. Поэтому в load_roads ниже явно передаётся filter_type='keep'.
# Иначе фильтр не оставляет перечисленные дороги, а исключает их.

ROAD_HIGHWAY = [
    'construction', 'crossing', 'living_street', 'motorway', 'motorway_link',
    'motorway_junction', 'pedestrian', 'primary', 'primary_link', 'raceway',
    'residential', 'road', 'secondary', 'secondary_link', 'service', 'services',
    'tertiary', 'tertiary_link', 'track', 'trunk', 'trunk_link',
    'turning_circle', 'turning_loop', 'unclassified',
]

# Значения service, которые полезно видеть в аудите. Это НЕ отдельный фильтр:
# все highway=service уже включены в ROAD_HIGHWAY выше.
ROAD_SERVICE_AUDIT = ['driveway', 'parking_aisle', 'alley', 'emergency_access', 'living_street']
ROAD_SERVICE = ROAD_SERVICE_AUDIT  # оставлено для диагностических распечаток

# Железные дороги: только действующие рельсовые пути.
RAILWAY_FILTER = {'railway': ['rail']}

# Вода: полигоны (озёра, заливы, водохранилища) и линии (реки, каналы, канавы).
WATER_FILTER = [
    '["natural"~"^(water|bay|wetland)$"]',
    '["waterway"~"^(river|canal|ditch|stream|riverbank)$"]',
    '["water"~"."]',
    '["landuse"~"^(basin|reservoir)$"]',
]

print('Наборы объектов OSM заданы.')


In [ ]:
# Чтение слоёв OSM из локального архива (при повторном запуске читаются из файла).
from pyrosm import OSM

_osm = None


def _get_osm():
    """Один объект pyrosm на весь ноутбук: он кэширует разобранный архив."""
    global _osm
    if _osm is None:
        _osm = OSM(OSM_PBF_PATH)
    return _osm


def _empty_gdf():
    return gpd.GeoDataFrame(geometry=[], crs=4326)


def _clean_layer(gdf, allowed_types, name, make_valid=False, keep_columns=None):
    """Убирает пустые/невалидные геометрии и оставляет нужные типы и колонки."""
    if gdf is None or len(gdf) == 0:
        print(f'{name}: объектов нет')
        return _empty_gdf()
    gdf = gdf.copy()
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty]
    if make_valid:
        bad = ~gdf.geometry.is_valid
        if bad.any():
            gdf.loc[bad, 'geometry'] = gdf.loc[bad, 'geometry'].make_valid()
    gdf = gdf[gdf.geom_type.isin(allowed_types)].copy()
    cols = ['geometry'] + [c for c in (keep_columns or []) if c in gdf.columns]
    gdf = gdf[cols].reset_index(drop=True)
    print(f'{name}: {len(gdf)} объектов')
    return gdf


def _check_layer_in_city(gdf, name):
    if len(gdf) == 0:
        raise RuntimeError(f'Слой «{name}» пуст. Проверьте границы и локальный архив OSM.')
    boundary_4326 = boundaries.to_crs(4326).union_all()
    n_inside = int(gdf.intersects(boundary_4326).sum())
    print(f'  {name}: пересекается с границей города — {n_inside} из {len(gdf)}')
    if n_inside == 0:
        raise RuntimeError(f'Слой «{name}» не пересекается с границей Тюмени. Проверьте bbox и PBF.')


def load_roads():
    if gdf_exists('roads'):
        return load_gdf('roads')
    print('Читаем дороги из локального архива OSM ...')
    roads = _get_osm().get_network(
        network_type='all', nodes=False,
        custom_filter={'highway': ROAD_HIGHWAY},
        filter_type='keep',                 # иначе dict-фильтр работает как exclude
        extra_attributes=['service', 'name'],
    )
    roads = _clean_layer(
        roads, ['LineString', 'MultiLineString'], 'Дороги',
        keep_columns=['highway', 'service', 'name'],
    )
    _check_layer_in_city(roads, 'Дороги')
    save_gdf(roads, 'roads')
    return roads


def load_railways():
    if gdf_exists('railways'):
        return load_gdf('railways')
    print('Читаем железные дороги из локального архива OSM ...')
    railway = _get_osm().get_data_by_custom_criteria(custom_filter=RAILWAY_FILTER)
    railway = _clean_layer(railway, ['LineString', 'MultiLineString'], 'Железные дороги')
    _check_layer_in_city(railway, 'Железные дороги')
    save_gdf(railway, 'railways')
    return railway


def load_water():
    if gdf_exists('water'):
        return load_gdf('water')
    print('Читаем воду из локального архива OSM ...')
    water = _get_osm().get_data_by_custom_criteria(custom_filter=WATER_FILTER)
    water = _clean_layer(
        water,
        ['LineString', 'MultiLineString', 'Polygon', 'MultiPolygon'],
        'Вода',
        make_valid=True,
    )
    _check_layer_in_city(water, 'Вода')
    save_gdf(water, 'water')
    return water


roads = load_roads()
railways = load_railways()
water = load_water()

print('Дорог:', len(roads), '| Ж/д:', len(railways), '| Вода:', len(water))
if 'highway' in roads.columns:
    print('Дороги по highway:')
    print(roads['highway'].value_counts().head(15).to_string())


In [ ]:
# Переводим слои в метрическую проекцию (метры) — так удобнее считать площади и нарезать кварталы
roads = roads.to_crs(utm)
railways = railways.to_crs(utm)
water = water.to_crs(utm)
boundaries_m = boundaries.to_crs(utm)
print('Слои переведены в проекцию', utm)

## 3. Здания: загрузка до нарезки

Здания нужны не только для населения, но и для дополнительного разделения
крупных кварталов. Поэтому они загружаются **до** нарезки.

Используется уже подготовленный `buildings_raw.geojson` (исходные контуры OSM),
если он есть в `refined/`. Иначе здания читаются из локального PBF.


In [ ]:
# Здания из локального архива OSM.
#
# Overpass не используется: архив Geofabrik уже обрезан по территории Тюмени
# с буфером 3 км. Оставляем только здания, чья внутренняя точка попадает
# в официальную границу города.

def load_buildings():
    if gdf_exists('buildings_raw'):
        return load_gdf('buildings_raw')
    if gdf_exists('buildings'):
        return load_gdf('buildings')

    print('Читаем здания из локального архива OSM ...')
    part = _get_osm().get_buildings(extra_attributes=['building:levels'])
    if part is None or len(part) == 0:
        raise RuntimeError('В архиве OSM не найдено ни одного здания. Проверьте границы территории.')

    part = part[part.geom_type.isin(['Polygon', 'MultiPolygon'])].copy()
    part = part[part.geometry.notna() & ~part.geometry.is_empty].copy()

    boundary_4326 = boundaries.to_crs(4326).union_all()
    inside = part.geometry.representative_point().within(boundary_4326)
    print('Зданий в буфере:', len(part), '| внутри границ города:', int(inside.sum()))
    part = part[inside.values].copy()

    keep = [c for c in ['building', 'building:levels', 'geometry'] if c in part.columns]
    part = part[keep].to_crs(utm).reset_index(drop=True)

    if 'building' in part.columns:
        part['building'] = part['building'].astype(str)

    if len(part) == 0:
        raise RuntimeError('После фильтрации по границе не осталось ни одного здания.')
    save_gdf(part, 'buildings_raw')
    save_gdf(part, 'buildings')
    return part


buildings = load_buildings()
print('Зданий:', len(buildings))
memory_report('после зданий')

In [ ]:
# Признак жилого здания и этажность
is_living_tags = ['residential', 'house', 'apartments', 'detached', 'terrace', 'dormitory']
if 'building' in buildings.columns:
    buildings['is_living'] = buildings['building'].apply(
        lambda v: str(v) in is_living_tags if pd.notna(v) else False
    )
else:
    buildings['is_living'] = False

if 'building:levels' in buildings.columns:
    buildings['number_of_floors'] = pd.to_numeric(buildings['building:levels'], errors='coerce')
else:
    buildings['number_of_floors'] = None

print('Жилых зданий:', int(buildings['is_living'].sum()), 'из', len(buildings))

# Первые строки таблицы зданий (как buildings.sample(5) в примере)
buildings.head()

In [ ]:
# Восстанавливаем пропущенные характеристики зданий
from blocksnet.preprocessing.imputing import impute_buildings

buildings = impute_buildings(buildings, default_living_demand=30)
save_gdf(buildings, 'buildings')
print('Суммарное население по зданиям:', int(buildings['population'].sum()))


## 4. Базовая нарезка кварталов

Дороги, железные дороги и вода переводятся в линии/полигоны, затем город
нарезается на кварталы. Микро-кварталы меньше `MIN_BLOCK_AREA_M2` удаляются.

Базовая нарезка сохраняется в `blocks_initial.geojson`. Дополнительное
разделение крупных кварталов выполняется в следующем разделе и сохраняется
в `blocks_refined.geojson` / `blocks.geojson`.


In [ ]:
# Шаг 1. Разделяем объекты на линии (дороги, ж/д) и полигоны (вода)
from blocksnet.blocks.cutting import preprocess_urban_objects, cut_urban_blocks

lines, polygons = preprocess_urban_objects(roads, railways, water)
print('Линий:', len(lines), '| Полигонов:', len(polygons))

In [ ]:
# Базовая нарезка кварталов (до дополнительного split).
if gdf_exists('blocks_initial'):
    blocks = load_gdf('blocks_initial')
else:
    raw_blocks = cut_urban_blocks(boundaries_m, lines, polygons)
    raw_blocks = raw_blocks.reset_index(drop=True)
    n_raw = len(raw_blocks)
    blocks = raw_blocks[raw_blocks.area >= MIN_BLOCK_AREA_M2].copy().reset_index(drop=True)
    print(f'Нарезано кварталов: {n_raw} | после фильтра < {MIN_BLOCK_AREA_M2:g} м²: {len(blocks)} '
          f'(удалено {n_raw - len(blocks)})')
    save_gdf(blocks, 'blocks_initial')

blocks_initial = blocks.copy()

if len(blocks) == 0:
    raise RuntimeError('Не осталось ни одного квартала. Уменьшите MIN_BLOCK_AREA_M2.')

if not blocks.geometry.is_valid.all():
    bad = ~blocks.geometry.is_valid
    blocks.loc[bad, 'geometry'] = blocks.loc[bad, 'geometry'].make_valid()
    blocks = blocks[blocks.geom_type.isin(['Polygon', 'MultiPolygon'])].reset_index(drop=True)
    blocks_initial = blocks.copy()
    save_gdf(blocks_initial, 'blocks_initial')
    print('Исправлены невалидные геометрии кварталов.')

coverage = blocks.area.sum() / boundaries_m.area.sum()
print('Кварталов до split:', len(blocks))
print('Суммарная площадь кварталов, га:', round(blocks.area.sum() / 10000, 1))
print('Доля площади города, покрытая кварталами:', round(float(coverage) * 100, 2), '%')
if coverage < 0.85:
    raise RuntimeError(
        f'Кварталы покрывают только {coverage * 100:.1f}% площади города. '
        'Проверьте дороги/воду и границы.'
    )


## 5. Дополнительное разделение крупных кварталов

Порог `SPLIT_MAX_AREA_KM2` задаёт, какие кварталы считать крупными. Кварталы
с застройкой больше порога передаются в `blocksnet.blocks.cutting.processing.split.split_blocks`.

Внутри BlocksNet:

1. находит крупные кварталы с зданиями;
2. кластеризует здания;
3. строит выпуклые оболочки кластеров;
4. продлевает существующие линии и добавляет границы кластеров;
5. повторно режет квартал и объединяет пустые/невалидные фрагменты.

Используется простая **площадная классификация** (`AreaClassifier`): квартал
с площадью больше порога помечается как `large`, остальные — как `normal`.
Это замена ML-классификатору `BlocksClassifier.default()`, для которого нужен
`pygeoops`. Если `pygeoops` установлен, можно поставить
`USE_ML_CLASSIFIER = True` и использовать штатный классификатор.

> В текущем окружении нет пакета `hdbscan`, который ожидает штатная
> кластеризация BlocksNet. Ноутбук автоматически использует
> `sklearn.cluster.HDBSCAN` только для кластеризации зданий; остальные шаги
> `split_blocks` остаются штатными. Если `hdbscan` установить, будет выбран он.

> Крупные полигоны без зданий (лес, промзоны, пустая периферия) не дробятся
> искусственно: у них нет внутренней структуры, которую можно восстановить по
> зданиям. Они остаются крупными, но их вклад в население близок к нулю.


In [ ]:
# Дополнительное разделение крупных кварталов.
from blocksnet.blocks.cutting.processing.split import split_blocks
from blocksnet.enums import BlockCategory
import blocksnet.blocks.cutting.processing.split as _split_mod

# BlocksNet ожидает пакет hdbscan. В этом окружении его нет, поэтому при
# необходимости подменяем только функцию кластеризации зданий на sklearn.HDBSCAN.
# Остальные шаги split_blocks (кластеры, выпуклые оболочки, продление линий,
# повторная нарезка) остаются штатными.
try:
    import hdbscan  # noqa: F401
    _cluster_backend = 'hdbscan'
except ImportError:
    from sklearn.cluster import HDBSCAN

    def _clusterize_for_split(buildings_gdf):
        gdf = buildings_gdf.copy()
        if len(gdf) == 0:
            gdf['cluster'] = []
            return gdf
        centers = gdf.geometry.centroid
        coords = np.column_stack([centers.x.values, centers.y.values])
        labels = np.full(len(gdf), -1, dtype=int)
        try:
            labels = HDBSCAN(min_cluster_size=10).fit_predict(coords)
        except Exception:
            labels = np.full(len(gdf), -1, dtype=int)
        # Если HDBSCAN не нашёл ни одного кластера, делим на 2–4 компактные группы.
        if len(set(labels) - {-1}) < 2:
            from sklearn.cluster import KMeans
            k = min(4, max(2, len(gdf) // 10))
            if len(gdf) >= 2:
                labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(coords)
        gdf['center'] = centers
        gdf['cluster'] = labels
        return gdf

    _split_mod.clusterize = _clusterize_for_split
    _cluster_backend = 'sklearn.HDBSCAN'


class AreaClassifier:
    """Минимальная замена BlocksClassifier: large при площади > порога.

    Возвращает таблицу с теми же колонками, которые ожидает split_blocks:
    category, invalid, normal, large. ML-модель и pygeoops не нужны.
    """

    def __init__(self, max_area_m2):
        self.max_area_m2 = float(max_area_m2)

    def run(self, blocks_gdf):
        area = blocks_gdf.geometry.area
        category = np.where(area > self.max_area_m2, BlockCategory.LARGE, BlockCategory.NORMAL)
        out = pd.DataFrame(index=blocks_gdf.index)
        out['category'] = category
        out['invalid'] = (category == BlockCategory.INVALID).astype(float)
        out['normal'] = (category == BlockCategory.NORMAL).astype(float)
        out['large'] = (category == BlockCategory.LARGE).astype(float)
        return out


def _geom_signature(gdf):
    """Короткая подпись геометрии кварталов для проверки кэша."""
    values = np.column_stack([
        np.asarray(gdf.geometry.bounds, dtype='float64'),
        np.asarray(gdf.geometry.area, dtype='float64'),
    ])
    return hashlib.md5(np.round(values, 3).tobytes()).hexdigest()


def _invalidate_dependent():
    """Удаляет только производные файлы refined, зависящие от кварталов."""
    names = [
        'blocks_land_use', 'buildings_blocks', 'schools', 'schools_blocks',
        'provision_schools', 'provision_links', 'density_indicators',
        'morphotypes_spacematrix', 'morphotypes_spacematrix_clusters',
        'morphotypes_strelka', 'area_accessibility', 'results_blocks',
        'results_table', 'results_summary', 'matrix_meta',
        'accessibility_matrix', 'comparison', 'quality_report',
    ]
    for name in names:
        for ext in ('.geojson', '.csv', '.parquet', '.json'):
            p = DATA_DIR / f'{name}{ext}'
            if p.exists():
                p.unlink()
    part_dir = DATA_DIR / 'accessibility_matrix_parts'
    if part_dir.exists():
        for p in part_dir.glob('part_*.parquet'):
            p.unlink()
    print('Производные файлы refined, зависящие от старой нарезки, удалены.')


split_params = {
    'method': 'area_classifier' if not USE_ML_CLASSIFIER else 'blocks_classifier_default',
    'cluster_backend': _cluster_backend,
    'max_area_km2': float(SPLIT_MAX_AREA_KM2),
    'min_block_area_m2': float(MIN_BLOCK_AREA_M2),
}
print('Кластеризация зданий для split:', _cluster_backend)
input_sig = _geom_signature(blocks_initial)
cached_meta = load_json('split_meta')
use_cache = (
    SPLIT_LARGE_BLOCKS
    and gdf_exists('blocks_refined')
    and bool(cached_meta)
    and cached_meta.get('params') == split_params
    and cached_meta.get('input_signature') == input_sig
    and cached_meta.get('input_blocks') == len(blocks_initial)
)

if not SPLIT_LARGE_BLOCKS:
    blocks = blocks_initial.copy()
    _invalidate_dependent()
    print('SPLIT_LARGE_BLOCKS = False: используется базовая нарезка.')
elif use_cache:
    blocks = load_gdf('blocks_refined')
    print('Дополнительное разделение прочитано из кэша: blocks_refined.geojson')
else:
    print(f'Кандидатов до split: {len(blocks_initial)} кварталов; порог {SPLIT_MAX_AREA_KM2:g} км².')
    if USE_ML_CLASSIFIER:
        from blocksnet.blocks.classification import BlocksClassifier
        classifier = BlocksClassifier.default()
    else:
        classifier = AreaClassifier(SPLIT_MAX_AREA_M2)

    try:
        blocks = split_blocks(
            blocks_initial, lines, polygons, buildings,
            classifier=classifier,
        )
    except Exception as exc:
        print('split_blocks не сработал, оставляем базовую нарезку:', exc)
        blocks = blocks_initial.copy()

    # В split-фрагментах появляются служебные колонки классификатора.
    drop_cols = [c for c in ['category', 'invalid', 'normal', 'large', 'has_buildings']
                 if c in blocks.columns]
    if drop_cols:
        blocks = blocks.drop(columns=drop_cols)

    blocks = blocks[blocks.geometry.notna() & ~blocks.geometry.is_empty].copy()
    blocks = blocks[blocks.geom_type.isin(['Polygon', 'MultiPolygon'])].copy()
    blocks = blocks[blocks.area >= MIN_BLOCK_AREA_M2].copy().reset_index(drop=True)
    if not blocks.geometry.is_valid.all():
        bad = ~blocks.geometry.is_valid
        blocks.loc[bad, 'geometry'] = blocks.loc[bad, 'geometry'].make_valid()
        blocks = blocks[blocks.geom_type.isin(['Polygon', 'MultiPolygon'])].reset_index(drop=True)

    _invalidate_dependent()
    save_gdf(blocks, 'blocks')
    save_gdf(blocks, 'blocks_refined')
    save_json({
        'params': split_params,
        'input_blocks': int(len(blocks_initial)),
        'input_signature': input_sig,
        'output_blocks': int(len(blocks)),
        'output_signature': _geom_signature(blocks),
        'created_at_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
        'note': 'area classifier; empty large blocks without buildings are not split',
    }, 'split_meta')
    print('После split:', len(blocks), 'кварталов')

print('Итоговая нарезка:', len(blocks), 'кварталов')
print('Максимальная площадь, км²:', round(float(blocks.area.max() / 1e6), 3))


In [ ]:
# Проверки геометрии новой нарезки.
import shapely


def _hole_count(geom):
    if geom.geom_type == 'Polygon':
        return len(geom.interiors)
    if geom.geom_type == 'MultiPolygon':
        return sum(len(p.interiors) for p in geom.geoms)
    return 0


qa = blocks[['geometry']].copy()
qa['area_km2'] = qa.area / 1e6
qa['holes'] = qa.geometry.apply(_hole_count)
qa['vertices'] = qa.geometry.apply(lambda g: shapely.get_num_coordinates(g))
qa['compactness'] = 4 * np.pi * qa.area / (qa.length ** 2).replace(0, np.nan)

city_area = boundaries_m.area.sum() / 1e6
initial_area = blocks_initial.area.sum() / 1e6
area_change = None
water_poly = water[water.geom_type.isin(['Polygon', 'MultiPolygon'])]
water_inside = water_poly.union_all().intersection(boundaries_m.union_all()).area / 1e6
land_area = city_area - water_inside
blocks_area = qa.area.sum() / 1e6
area_change = blocks_area - initial_area
overlap_area = (qa.area.sum() - qa.union_all().area) / 1e6
outside_area = qa.geometry.difference(boundaries_m.union_all()).area.sum() / 1e6
invalid = int((~qa.geometry.is_valid).sum())
coverage_land = blocks_area / land_area * 100 if land_area else np.nan

top4 = qa.nlargest(4, 'area_km2')
print('Кварталов до split:', len(blocks_initial))
print('Кварталов после split:', len(qa))
print('Площадь кварталов, км²:', round(blocks_area, 2))
print('Изменение площади после split, км²:', round(float(area_change), 6))
print('Покрытие суши, %:', round(coverage_land, 2))
print('Максимальная площадь, км²:', round(float(qa['area_km2'].max()), 2))
print('Кварталов > 10 км²:', int((qa['area_km2'] > 10).sum()))
print('Кварталов с > 50 внутренних контуров:', int((qa['holes'] > 50).sum()))
print('Невалидных геометрий:', invalid)
print('Площадь пересечений кварталов, км²:', round(float(overlap_area), 9))
print('Площадь кварталов вне границы, км²:', round(float(outside_area), 9))
print('Доля площади топ-4 кварталов, %:', round(float(top4['area_km2'].sum() / blocks_area * 100), 1))

if invalid:
    raise RuntimeError('После split остались невалидные геометрии кварталов.')
if overlap_area > 0.01:
    raise RuntimeError(f'Слишком большая площадь пересечений кварталов: {overlap_area:.6f} км²')
if outside_area > 0.01:
    raise RuntimeError(f'Слишком большая площадь кварталов вне границы: {outside_area:.6f} км²')
if coverage_land < 98.0:
    raise RuntimeError(f'Кварталы покрывают только {coverage_land:.2f}% суши.')
if abs(area_change) > 0.1:
    raise RuntimeError(f'Площадь кварталов изменилась после split на {area_change:.3f} км².')

save_df(qa.drop(columns='geometry'), 'blocks_quality')
save_json({
    'blocks_before_split': int(len(blocks_initial)),
    'blocks_after_split': int(len(qa)),
    'blocks_area_km2': round(float(blocks_area), 3),
    'area_change_km2': round(float(area_change), 6),
    'land_area_km2': round(float(land_area), 3),
    'coverage_land_pct': round(float(coverage_land), 3),
    'max_area_km2': round(float(qa['area_km2'].max()), 3),
    'blocks_gt_10_km2': int((qa['area_km2'] > 10).sum()),
    'blocks_holes_gt_50': int((qa['holes'] > 50).sum()),
    'invalid_geometries': invalid,
    'overlap_area_km2': round(float(overlap_area), 9),
    'outside_area_km2': round(float(outside_area), 9),
    'top4_area_share_pct': round(float(top4['area_km2'].sum() / blocks_area * 100), 2),
}, 'blocks_quality')
display(qa.nlargest(15, 'area_km2').round(2))


In [ ]:
# Карта итоговых кварталов
ax = blocks.plot(figsize=(10, 10), color='#eeeeee', edgecolor='#555555', linewidth=0.4)
ax.set_title(f'Кварталы после дополнительного split: {len(blocks)}')
ax.set_axis_off()
plt.show()


## 6. Землепользование кварталов

Берём функциональные зоны OSM (`landuse`) и сопоставляем их с типами землепользования BlocksNet.


In [ ]:
# Функциональные зоны из OpenStreetMap (landuse) — из локального архива
def load_functional_zones():
    if gdf_exists('functional_zones'):
        return load_gdf('functional_zones')

    print('Читаем функциональные зоны (landuse) из локального архива OSM ...')
    fz = _get_osm().get_landuse()

    if fz is None or len(fz) == 0:
        raise RuntimeError('В архиве OSM не найдено функциональных зон (landuse).')

    fz = fz.copy()
    fz = fz[fz.geometry.notna() & ~fz.geometry.is_empty]
    bad = ~fz.geometry.is_valid
    if bad.any():
        fz.loc[bad, 'geometry'] = fz.loc[bad, 'geometry'].make_valid()
    fz = fz[fz.geom_type.isin(['Polygon', 'MultiPolygon'])].copy()

    boundary_4326 = boundaries.to_crs(4326).union_all()
    fz = fz[fz.intersects(boundary_4326)].copy()

    fz = fz[['geometry', 'landuse']].rename(columns={'landuse': 'functional_zone'})
    fz = fz.to_crs(utm)
    fz['functional_zone'] = fz['functional_zone'].astype(str)
    fz = fz.reset_index(drop=True)
    save_gdf(fz, 'functional_zones')
    return fz


functional_zones = load_functional_zones()
print('Функциональных зон:', len(functional_zones))
functional_zones[['functional_zone']].head()

In [ ]:
# Правила: какой функциональной зоне OSM соответствует тип землепользования
from blocksnet.enums import LandUse
from blocksnet.blocks.assignment import assign_land_use

rules = {
    'commercial': LandUse.BUSINESS,
    'industrial': LandUse.INDUSTRIAL,
    'cemetery': LandUse.SPECIAL,
    'garages': LandUse.INDUSTRIAL,
    'residential': LandUse.RESIDENTIAL,
    'retail': LandUse.BUSINESS,
    'grass': LandUse.RECREATION,
    'farmland': LandUse.AGRICULTURE,
    'construction': LandUse.SPECIAL,
    'brownfield': LandUse.INDUSTRIAL,
    'forest': LandUse.RECREATION,
    'recreation_ground': LandUse.RECREATION,
    'religious': LandUse.SPECIAL,
    'flowerbed': LandUse.RECREATION,
    'military': LandUse.SPECIAL,
    'landfill': LandUse.TRANSPORT,
}

# Если слой землепользования уже посчитан — читаем его из файла.
# Если вы изменили правила выше, удалите blocks_land_use.geojson,
# чтобы слой пересчитался.
if gdf_exists('blocks_land_use'):
    blocks = load_gdf('blocks_land_use')
else:
    blocks = assign_land_use(blocks, functional_zones, rules)
    # land_use хранится как перечисление — переводим в текст, чтобы сохранить в GeoJSON
    blocks['land_use'] = blocks['land_use'].apply(lambda v: None if v is None else v.value)
    save_gdf(blocks, 'blocks_land_use')

ax = blocks.plot(figsize=(10, 10), color='#dddddd', edgecolor='#888888', linewidth=0.3)
blocks.plot(column='land_use', categorical=True, legend=True, ax=ax,
            edgecolor='#888888', linewidth=0.3)
ax.set_title('Землепользование кварталов')
ax.set_axis_off()
plt.show()

# Таблица долей землепользования по кварталам (как blocks.head() в примере)
blocks.head()


## 7. Пешеходный граф и матрица доступности

**Какой граф используется.** Готового файла графа именно для Тюмени в открытом доступе нет. Поэтому берётся **стандартный пешеходный граф OpenStreetMap** для нашей территории: он собирается **из локального архива OSM** библиотекой `pyrosm` и приводится к формату, который ждёт BlocksNet.

- **Источник:** OpenStreetMap (локальный архив Geofabrik, без Overpass API).
- **Способ передвижения:** пешком, скорость около 5 км/ч.
- **Время:** в минутах (атрибут рёбер `time_min`).
- **Интермодальный граф не строим.**

> **Почему матрица считается порциями.** В Тюмени после нарезки около 10–14 тысяч кварталов. Полная матрица N×N — это сотни миллионов ячеек. Библиотека при одном вызове `calculate_accessibility_matrix` держит на пике сразу несколько полных копий матрицы, а numba дополнительно выделяет рабочие массивы на каждый поток. При занятой оперативной памяти ядро Jupyter и VS Code могут завершиться без обычной ошибки Python (именно это случилось в прошлом исследовании Тюмени). Поэтому матрица считается **порциями по строкам**, каждая порция сразу сохраняется на диск, а после сбоя расчёт продолжается с последней сохранённой порции.

> Размер порции и число потоков задаются в ячейке настроек (`MATRIX_CHUNK_SIZE`, `MATRIX_WORKERS`). По умолчанию — 100 строк и 2 потока. Если свободной памяти меньше 2 ГБ, ноутбук автоматически уменьшит порцию до 50 и оставит 1 поток. Порции хранятся в Parquet, итоговая матрица — в Parquet и (по желанию) в CSV. Слои графа (`graph_nodes`, `graph_edges`) по умолчанию не сохраняются: для показателей они не нужны. Если нужны карты графа, поставьте `SAVE_GRAPH_LAYERS = True`.

> После расчёта матрицы граф удаляется из памяти — он больше не нужен, а освободившиеся ~1 ГБ используются для агрегации зданий и расчёта показателей.


In [ ]:
# Пешеходный граф из локального архива OSM (при повторном запуске читается из файла).
import networkx as nx
from iduedu.utils.utils import remove_weakly_connected_nodes  # логика BlocksNet: оставить одну связную часть


def _walk_graph_from_local_pbf(walk_speed=5 * 1000 / 60):
    """Собирает пешеходный граф в формате, который ждёт BlocksNet / IduEdu.

    Отличие от get_accessibility_graph: граф строится не из Overpass, а из
    локального архива OSM (pyrosm). Формат результата тот же: узлы с x/y в
    метрической проекции, рёбра с length_meter / time_min, graph['crs'], graph['type'].
    """
    nodes, edges = _get_osm().get_network(network_type='walking', nodes=True)
    graph = _get_osm().to_graph(
        nodes, edges, graph_type='networkx',
        network_type='walking', retain_all=False, simplify=True,
    )
    del nodes, edges
    gc.collect()

    graph.graph['crs'] = 'epsg:4326'
    nodes_gdf, edges_gdf = ox.graph_to_gdfs(graph)
    del graph
    gc.collect()

    utm_epsg = int(nodes_gdf.estimate_utm_crs().to_epsg())
    nodes_gdf = nodes_gdf.to_crs(utm_epsg)
    edges_gdf = edges_gdf.to_crs(utm_epsg)

    nodes_gdf['x'] = nodes_gdf.geometry.x
    nodes_gdf['y'] = nodes_gdf.geometry.y
    edges_gdf['length_meter'] = edges_gdf.geometry.length.round(3)
    edges_gdf['time_min'] = (edges_gdf.geometry.length / walk_speed).round(3)
    edges_gdf['type'] = 'walk'

    walk_graph = ox.graph_from_gdfs(
        nodes_gdf[['x', 'y', 'geometry']],
        edges_gdf[['length_meter', 'time_min', 'type', 'geometry']],
    )
    del nodes_gdf, edges_gdf
    gc.collect()

    # Оставляем одну связную часть и перенумеровываем узлы в 0..N-1 —
    # именно этого ждёт расчёт матрицы доступности в BlocksNet.
    walk_graph = remove_weakly_connected_nodes(walk_graph)
    walk_graph = nx.relabel_nodes(walk_graph, {old: new for new, old in enumerate(walk_graph.nodes())})
    walk_graph.graph['crs'] = utm_epsg
    walk_graph.graph['type'] = 'walk'
    walk_graph.graph['walk_speed'] = walk_speed
    return walk_graph


def get_walk_graph():
    if graph_exists('graph'):
        return load_graph('graph')
    print('Собираем пешеходный граф из локального архива OSM (может занять несколько минут) ...')
    graph = _walk_graph_from_local_pbf()
    save_graph(graph, 'graph')
    return graph


walk_graph = get_walk_graph()

gtype = walk_graph.graph.get('type')
gcrs = walk_graph.graph.get('crs')
print('Тип графа:', gtype, '| Проекция:', gcrs)
print('Узлов:', walk_graph.number_of_nodes(), '| Рёбер:', walk_graph.number_of_edges())
if walk_graph.number_of_nodes() < 1000:
    raise RuntimeError('Граф слишком маленький — проверьте границы территории и архив OSM.')
if not nx.is_weakly_connected(walk_graph):
    print('ВНИМАНИЕ: граф не является связным после удаления слабых компонент.')
memory_report('после построения графа')

In [ ]:
# Матрица доступности для новой квартальной структуры.
# Расчёт идёт порциями и сохраняется в data_tyumen/refined/accessibility_matrix_parts/.
#
# Важно: iduedu.get_adj_matrix_gdf_to_gdf на каждый вызов заново копирует граф,
# строит KDTree и scipy-матрицу. При 500+ порциях это даёт большой расход ОЗУ.
# Ниже эти подготовки выполняются один раз, а по порциям считается только
# numba-Дейкстра. Результат тот же, что у штатной функции.
import numpy as np
import numba as nb
from blocksnet.relations import validate_accessibility_graph
from iduedu.modules.matrix.matrix_builder import (
    dijkstra_numba_parallel,
    get_closest_nodes,
    _get_sparse_row,
    _get_numba_matrix_attr,
)
from iduedu.modules.matrix.numba_csr_matrix import UI32CSRMatrix

PART_DIR = DATA_DIR / 'accessibility_matrix_parts'
PART_DIR.mkdir(parents=True, exist_ok=True)

MATRIX_PARQUET = DATA_DIR / 'accessibility_matrix.parquet'
MATRIX_CSV = DATA_DIR / 'accessibility_matrix.csv'


# Если ядро перезапускали, для продолжения достаточно только кварталов и графа.
# Остальные слои (дороги, вода, здания) для матрицы не нужны.
if 'blocks' not in globals():
    if gdf_exists('blocks_land_use'):
        print('Загружаем blocks_land_use.geojson для продолжения матрицы ...')
        blocks = load_gdf('blocks_land_use')
    elif gdf_exists('blocks_refined'):
        print('Загружаем blocks_refined.geojson для продолжения матрицы ...')
        blocks = load_gdf('blocks_refined')
    else:
        raise RuntimeError(
            'Нет blocks_land_use.geojson или blocks_refined.geojson. '
            'Сначала выполните нарезку кварталов.'
        )
blocks.index = blocks.index.astype(int)
if 'utm' not in globals():
    utm = blocks.crs if getattr(blocks.crs, 'is_projected', False) else blocks.estimate_utm_crs()


def _blocks_signature(gdf):
    values = np.column_stack([
        np.asarray(gdf.geometry.bounds, dtype='float64'),
        np.asarray(gdf.geometry.area, dtype='float64'),
    ])
    return hashlib.md5(np.round(values, 3).tobytes()).hexdigest()


_blocks_sig = _blocks_signature(blocks)

# Отдельная подпись для незавершённых порций: если кварталы изменились,
# нельзя продолжать старую частичную матрицу.
PART_SIG_PATH = DATA_DIR / 'matrix_input_signature.txt'
if PART_SIG_PATH.exists() and PART_SIG_PATH.read_text(encoding='utf-8').strip() != _blocks_sig:
    print('Кварталы изменились: удаляем старые частичные порции refined.')
    for p in PART_DIR.glob('part_*.parquet'):
        p.unlink()
PART_SIG_PATH.write_text(_blocks_sig, encoding='utf-8')

# Если кварталы изменились, старые порции и матрица не должны переиспользоваться.
_cached_meta = load_json('matrix_meta')
if _cached_meta and _cached_meta.get('blocks_signature') != _blocks_sig:
    print('Кварталы изменились: удаляем устаревшую матрицу и порции refined.')
    for p in (MATRIX_PARQUET, MATRIX_CSV):
        p.unlink(missing_ok=True)
    for p in PART_DIR.glob('part_*.parquet'):
        p.unlink()
    _cached_meta = None

for tmp_path in PART_DIR.glob('*.tmp'):
    tmp_path.unlink()

# Крупные слои, которые уже не нужны для матрицы. Здания будут перечитаны
# из buildings.geojson перед агрегацией, поэтому их тоже можно освободить.
for _layer_name in (
    'raw_blocks', 'blocks_initial', 'qa', 'top4',
    'roads', 'railways', 'water', 'functional_zones', 'lines', 'polygons',
    'buildings',
):
    if _layer_name in globals():
        del globals()[_layer_name]
gc.collect()

# На малой ОЗУ считаем маленькими порциями в один поток: так меньше пик памяти.
try:
    _free_now = psutil.virtual_memory().available / 1e9 if psutil is not None else None
except Exception:
    _free_now = None
if _free_now is not None and _free_now < 2.0:
    MATRIX_CHUNK_SIZE = min(MATRIX_CHUNK_SIZE, 25)
    MATRIX_WORKERS = 1
    print(f'Мало ОЗУ ({_free_now:.2f} ГБ): порция {MATRIX_CHUNK_SIZE} строк, 1 поток.')

# Граф ещё нужен для расчёта матрицы, но кэш pyrosm — уже нет.
release_osm_cache()
gc.collect()
memory_report('перед матрицей')


def _load_cached_matrix():
    """Читает готовую матрицу только при совпадении подписи кварталов."""
    meta = load_json('matrix_meta')
    if not meta or meta.get('blocks_signature') != _blocks_sig:
        return None
    for path in (MATRIX_PARQUET, MATRIX_CSV):
        if not path.exists():
            continue
        try:
            if path.suffix == '.parquet':
                matrix = pd.read_parquet(path)
            else:
                matrix = pd.read_csv(path, index_col=0, dtype=np.float16)
            matrix.index = matrix.index.astype(int)
            matrix.columns = matrix.columns.astype(int)
            if (matrix.shape == (len(blocks), len(blocks))
                    and matrix.index.equals(blocks.index)
                    and matrix.columns.equals(blocks.index)):
                return matrix.astype(np.float16)
            print('Файл матрицы не соответствует текущим кварталам:', path.name)
        except Exception as exc:
            print('Не удалось прочитать', path.name, '— пересчитаем порциями:', exc)
    return None


acc_mx = _load_cached_matrix()

if acc_mx is None:
    done_ids = set()
    for part_path in sorted(PART_DIR.glob('part_*.parquet')):
        try:
            part = pd.read_parquet(part_path)
            part.index = part.index.astype(int)
            part.columns = part.columns.astype(int)
            if (len(part.columns) != len(blocks)
                    or not part.columns.equals(blocks.index)):
                print('Порция не соответствует текущим кварталам, удаляем:', part_path.name)
                part_path.unlink()
                continue
            if not part.index.isin(blocks.index).all():
                print('Порция содержит чужие id кварталов, удаляем:', part_path.name)
                part_path.unlink()
                continue
            done_ids.update(part.index.tolist())
            del part
        except Exception as exc:
            print('Повреждённая порция, удаляем:', part_path.name, '|', exc)
            part_path.unlink()

    remaining_ids = [i for i in blocks.index if i not in done_ids]
    print(f'Уже посчитано строк: {len(done_ids)} | осталось: {len(remaining_ids)}')

    _graph_nodes = None
    _graph_edges = None

    if remaining_ids:
        if 'walk_graph' not in globals():
            if not graph_exists('graph'):
                raise RuntimeError(
                    'Нет graph.pkl. Сначала выполните ячейку построения графа '
                    'или вернитесь к полному порядку запуска.'
                )
            print('Загружаем graph.pkl для продолжения матрицы ...')
            walk_graph = load_graph('graph')

        validate_accessibility_graph(walk_graph)

        available = psutil.virtual_memory().available / 1e9 if psutil is not None else None
        if available is not None and available < 0.5:
            raise RuntimeError(
                f'Свободно всего {available:.2f} ГБ. '
                'Перезапустите ядро, затем выполните только ячейки '
                '«Импорты и настройки», «Вспомогательные функции» и эту ячейку. '
                'Готовые порции сохранены, расчёт продолжится с них.'
            )

        print('Готовим граф и ближайшие узлы один раз для всех кварталов ...')
        closest_all, dist_all = get_closest_nodes(blocks, walk_graph)
        closest_all = np.asarray(closest_all, dtype=np.int32)
        dist_all = np.asarray(dist_all, dtype=np.float64)

        print('Строим разреженную матрицу графа один раз ...')
        sparse_row = _get_sparse_row(walk_graph, 'time_min')
        csr_matrix = UI32CSRMatrix(*_get_numba_matrix_attr(sparse_row))
        del sparse_row
        gc.collect()

        _graph_nodes = int(walk_graph.number_of_nodes())
        _graph_edges = int(walk_graph.number_of_edges())

        # Дальше граф не нужен: все расстояния считаются по csr_matrix.
        # Это освобождает сотни мегабайт до начала расчёта порций.
        if not SAVE_GRAPH_LAYERS:
            del walk_graph
            gc.collect()
            print('Граф удалён из памяти: дальше используется только csr-матрица.')

        nb.set_num_threads(MATRIX_WORKERS)
        dijkstra_numba_parallel(
            numba_matrix=csr_matrix,
            sources=closest_all[:1].copy(),
            targets=closest_all[:1].copy(),
            cutoff=np.int32(0),
        )
        memory_report('после подготовки матрицы')

        remaining_set = set(remaining_ids)
        t_start = time.time()
        for start in range(0, len(blocks), MATRIX_CHUNK_SIZE):
            chunk = blocks.iloc[start:start + MATRIX_CHUNK_SIZE]
            chunk = chunk.loc[chunk.index.isin(remaining_set)]
            if chunk.empty:
                continue

            if psutil is not None:
                available = psutil.virtual_memory().available / 1e9
                if available < 1.0:
                    gc.collect()
                    available = psutil.virtual_memory().available / 1e9
                    print(f'  Мало свободной ОЗУ ({available:.2f} ГБ) — собираем мусор.')
                # 0.2 ГБ достаточно для одной маленькой порции; жёсткий порог
                # 0.5 ГБ не давал продолжить расчёт даже после очистки памяти.
                if available < 0.2:
                    raise RuntimeError(
                        f'Слишком мало свободной ОЗУ ({available:.2f} ГБ). '
                        'Готовые порции сохранены. Закройте лишние программы, '
                        'уменьшите MATRIX_CHUNK_SIZE и запустите ячейку снова.'
                    )

            positions = blocks.index.get_indexer(chunk.index)
            if (positions < 0).any():
                raise RuntimeError('В порции оказались чужие id кварталов.')

            print(f'Считаем строки {int(chunk.index.min())}–{int(chunk.index.max())} '
                  f'({len(chunk)} из {len(remaining_ids)} оставшихся) ...')
            raw = dijkstra_numba_parallel(
                numba_matrix=csr_matrix,
                sources=closest_all[positions],
                targets=closest_all,
                cutoff=np.int32(np.iinfo(np.int32).max),
            )

            # Повторяем преобразования штатной функции iduedu:
            # сотые доли веса -> минуты + расстояние до/от ближайших узлов.
            values = raw.astype(np.float32) / 100.0
            # add_dist_tofrom_node=True, weight='time_min'
            speed = 5 * 1000 / 60
            additional = (dist_all[positions, None] + dist_all[None, :]) / speed
            mask = values > 0
            values = np.where(mask, values + additional, values)
            values = np.where(values >= 0, values, np.inf).astype(np.float16)

            part = pd.DataFrame(values, index=chunk.index, columns=blocks.index)
            part_path = PART_DIR / (
                f'part_{int(chunk.index.min()):06d}_{int(chunk.index.max()):06d}.parquet'
            )
            tmp_path = part_path.with_name(part_path.name + '.tmp')
            part.to_parquet(tmp_path)
            tmp_path.replace(part_path)
            print(f'  сохранено: {part_path.name}  {part.shape[0]} x {part.shape[1]}')
            del raw, values, part
            gc.collect()
        print('Расчёт порций занял', round(time.time() - t_start, 1), 'с')

        del csr_matrix, closest_all, dist_all
        gc.collect()
    else:
        # Все порции уже готовы; граф для сборки не нужен.
        if 'walk_graph' in globals():
            _graph_nodes = int(walk_graph.number_of_nodes())
            _graph_edges = int(walk_graph.number_of_edges())

    part_paths = sorted(PART_DIR.glob('part_*.parquet'))
    if not part_paths:
        raise RuntimeError('Не найдено ни одной готовой порции матрицы.')

    print(f'Собираем матрицу из {len(part_paths)} порций ...')
    n = len(blocks)

    # Граф больше не нужен: все строки посчитаны. Освобождаем его до
    # выделения полной матрицы n×n, иначе на сборке не хватает ОЗУ.
    if 'walk_graph' in globals():
        _graph_nodes = int(walk_graph.number_of_nodes())
        _graph_edges = int(walk_graph.number_of_edges())
        if not SAVE_GRAPH_LAYERS:
            del walk_graph
            gc.collect()
            print('Граф удалён перед сборкой матрицы — освобождена память.')

    matrix_array = np.full((n, n), np.nan, dtype=np.float16)
    for part_path in part_paths:
        part = pd.read_parquet(part_path)
        part.index = part.index.astype(int)
        part.columns = part.columns.astype(int)
        if not part.columns.equals(blocks.index):
            raise RuntimeError(f'Порция {part_path.name} имеет другой набор столбцов.')
        positions = blocks.index.get_indexer(part.index)
        if (positions < 0).any():
            raise RuntimeError(f'Порция {part_path.name} содержит чужие id кварталов.')
        matrix_array[positions, :] = part.to_numpy(dtype=np.float16)
        del part
        gc.collect()

    acc_mx = pd.DataFrame(matrix_array, index=blocks.index, columns=blocks.index, copy=False)
    del matrix_array
    gc.collect()

    empty_rows = acc_mx.index[acc_mx.isna().all(axis=1)].tolist()
    if empty_rows:
        raise RuntimeError(
            f'Не хватает данных для {len(empty_rows)} кварталов. '
            'Запустите ячейку ещё раз — расчёт продолжит с сохранённых порций.'
        )

    acc_mx.index = acc_mx.index.astype(int)
    acc_mx.columns = acc_mx.columns.astype(int)
    save_parquet(acc_mx, 'accessibility_matrix')

    if SAVE_MATRIX_CSV:
        estimated_gb = n * n * 6 / 1e9
        free_gb = shutil.disk_usage(DATA_DIR).free / 1e9
        if free_gb < estimated_gb + 1.5:
            print(f'CSV не сохраняем: свободно {free_gb:.1f} ГБ, нужно ~{estimated_gb:.1f} ГБ.')
        else:
            t0 = time.time()
            save_df(acc_mx, 'accessibility_matrix', float_format='%.2f')
            print('CSV сохранён за', round(time.time() - t0, 1), 'с')

    save_json({
        'shape': [int(n), int(n)],
        'chunk_size': int(MATRIX_CHUNK_SIZE),
        'workers': int(MATRIX_WORKERS),
        'graph_nodes': _graph_nodes,
        'graph_edges': _graph_edges,
        'units': 'minutes (walk)',
        'dtype': 'float16',
        'parts': len(part_paths),
        'blocks_signature': _blocks_sig,
        'created_at_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    }, 'matrix_meta')
    gc.collect()
else:
    print('Матрица прочитана из кэша (подпись кварталов совпала).')

print('Размер матрицы:', acc_mx.shape)
print('Значения — время в минутах пешком.')
print('Среднее время по диагонали (должно быть 0):', round(float(acc_mx.values.diagonal().mean()), 3))

if SAVE_GRAPH_LAYERS and not (gdf_exists('graph_nodes') and gdf_exists('graph_edges')):
    from blocksnet.relations import accessibility_graph_to_gdfs
    print('Сохраняем слои графа в GeoJSON ...')
    nodes_gdf, edges_gdf = accessibility_graph_to_gdfs(walk_graph)
    nodes_gdf = nodes_gdf[nodes_gdf.geometry.notna()]
    edges_gdf = edges_gdf[edges_gdf.geometry.notna()]
    save_gdf(nodes_gdf, 'graph_nodes')
    del nodes_gdf
    gc.collect()
    save_gdf(edges_gdf, 'graph_edges')
    del edges_gdf
    gc.collect()

if 'walk_graph' in globals():
    del walk_graph
    gc.collect()
    print('Граф удалён из памяти: матрица уже рассчитана.')
memory_report('после матрицы')


## 8. Агрегация зданий и население

Здания уже загружены и импутированы в разделе 3. Теперь они агрегируются по
**новым** кварталам. Население — модельная оценка по жилой площади зданий
(`default_living_demand=30`), а не официальная статистика.


In [ ]:
# Собираем здания в кварталы
from blocksnet.blocks.aggregation import aggregate_objects

# Матрица освобождает buildings из памяти; слой уже сохранён в GeoJSON.
if 'buildings' not in globals():
    if not gdf_exists('buildings'):
        raise RuntimeError('Нет buildings.geojson: сначала выполните ячейку обработки зданий.')
    print('Здания перечитаны из buildings.geojson после расчёта матрицы.')
    buildings = load_gdf('buildings')

buildings_blocks = aggregate_objects(blocks, buildings)[0]
print('Кварталов с данными по зданиям:', len(buildings_blocks))
print('Население (сумма):', int(buildings_blocks['population'].sum()))
save_gdf(buildings_blocks, 'buildings_blocks')

# Таблица агрегированных показателей по кварталам
buildings_blocks.head()

ax = blocks.plot(figsize=(10, 10), color='#eeeeee', edgecolor='#888888', linewidth=0.3)
buildings_blocks.plot(column='population', legend=True, ax=ax, cmap='Reds',
                      edgecolor='#888888', linewidth=0.2)
ax.set_title('Население по кварталам')
ax.set_axis_off()
plt.show()

## 9. Услуги: школы

Как и в оригинальном примере `pipeline.ipynb`, берём только школы. Школы читаются из того же локального архива OSM.


In [ ]:
# Школы из локального архива OSM
def load_schools():
    # schools_raw — исходные объекты до дедупликации; schools — уже обработанный слой.
    if gdf_exists('schools_raw'):
        return load_gdf('schools_raw')
    if gdf_exists('schools'):
        return load_gdf('schools')

    print('Читаем школы из локального архива OSM ...')
    s = _get_osm().get_data_by_custom_criteria(
        custom_filter={'amenity': ['school']},
        extra_attributes=['capacity', 'name', 'operator', 'ref', 'building'],
    )

    if s is None or len(s) == 0:
        raise RuntimeError('В архиве OSM не найдено ни одной школы. Проверьте границы территории.')

    s = s[s.geom_type.isin(['Polygon', 'MultiPolygon', 'Point', 'MultiPoint'])].copy()
    s = s[s.geometry.notna() & ~s.geometry.is_empty].copy()

    boundary_4326 = boundaries.to_crs(4326).union_all()
    inside = s.geometry.representative_point().within(boundary_4326)
    print('Школ в буфере:', len(s), '| внутри границ города:', int(inside.sum()))
    s = s[inside.values].copy()

    if 'id' in s.columns:
        s['osm_id'] = s['id'].astype(str)
    keep = [c for c in ['name', 'capacity', 'operator', 'ref', 'building', 'osm_id', 'geometry'] if c in s.columns]
    s = s[keep].to_crs(utm).reset_index(drop=True)
    s['capacity_osm'] = pd.to_numeric(s['capacity'], errors='coerce') if 'capacity' in s.columns else None

    save_gdf(s, 'schools_raw')
    save_gdf(s, 'schools')
    return s


schools = load_schools()
print('Школ (исходных объектов):', len(schools))
if len(schools) == 0:
    raise RuntimeError('После фильтрации по границе не осталось ни одной школы.')
if 'capacity' not in schools.columns:
    schools['capacity'] = None


In [ ]:
# Дедупликация школ, разделение фактической и модельной ёмкости, агрегация по кварталам.
import numpy as np
from blocksnet.blocks.aggregation import aggregate_objects

if 'capacity_source' not in schools.columns:
    schools = schools.copy().reset_index(drop=True)
    schools['capacity_osm'] = pd.to_numeric(schools['capacity'], errors='coerce') if 'capacity' in schools.columns else np.nan
    schools['name'] = schools['name'] if 'name' in schools.columns else None
    schools['name_norm'] = (
        schools['name'].fillna('').astype(str).str.lower()
        .str.replace('ё', 'е', regex=False)
        .str.replace(r'[^a-zа-я0-9]+', ' ', regex=True).str.strip()
    )

    # Объединяем объекты одного кампуса: одинаковое название в радиусе 300 м
    # или безымянный объект в радиусе 50 м от именованной школы.
    pts = schools.geometry.representative_point()
    xy = np.column_stack([pts.x.values, pts.y.values])
    dist = np.sqrt(((xy[:, None, :] - xy[None, :, :]) ** 2).sum(axis=2))
    parent = list(range(len(schools)))

    def _find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    def _union(a, b):
        ra, rb = _find(a), _find(b)
        if ra != rb:
            parent[rb] = ra

    for i in range(len(schools)):
        for j in range(i + 1, len(schools)):
            if dist[i, j] > 300:
                continue
            ni, nj = schools.at[i, 'name_norm'], schools.at[j, 'name_norm']
            same_name = bool(ni) and ni == nj
            one_named_near = bool(ni) != bool(nj) and dist[i, j] <= 50
            if same_name or one_named_near:
                _union(i, j)

    schools['group'] = [_find(i) for i in range(len(schools))]
    rows = []
    for root, idx in schools.groupby('group').groups.items():
        sub = schools.loc[list(idx)]
        names = [n for n in sub['name'].tolist() if pd.notna(n) and str(n).strip()]
        cap_osm = sub['capacity_osm'].dropna()
        cap_osm_val = float(cap_osm.max()) if len(cap_osm) else np.nan
        cap_model = 0.0 if np.isfinite(cap_osm_val) else 250.0
        areas = sub.geometry.area
        rep = int(areas.idxmax()) if float(areas.max()) > 0 else int(sub.index[0])
        rows.append({
            'name': names[0] if names else None,
            'name_norm': sub.loc[rep, 'name_norm'],
            'osm_ids': '|'.join(sub.get('osm_id', sub.get('id', pd.Series(index=sub.index, dtype=str))).astype(str).tolist()),
            'n_objects': len(sub),
            'capacity_osm': cap_osm_val,
            'capacity_modeled': cap_model,
            'capacity': cap_osm_val if np.isfinite(cap_osm_val) else cap_model,
            'capacity_source': 'osm' if np.isfinite(cap_osm_val) else 'modeled_250',
            'geometry': sub.loc[rep, 'geometry'],
        })
    schools = gpd.GeoDataFrame(rows, geometry='geometry', crs=schools.crs).reset_index(drop=True)
    save_gdf(schools, 'schools')
else:
    print('Школы уже дедуплицированы — используем сохранённый слой schools.')

schools_blocks = aggregate_objects(blocks, schools)[0]
save_gdf(schools_blocks, 'schools_blocks')

print('Школ после дедупликации:', len(schools))
print('Суммарная ёмкость (мест):', int(schools_blocks['capacity'].sum()))
print('Ёмкость из OSM:', int(schools['capacity_osm'].fillna(0).sum()),
      '| модельная 250:', int(schools['capacity_modeled'].sum()))
display(schools.head())
display(schools_blocks.head())


## 10. Показатели (как в `pipeline.ipynb`)

Считаем обеспеченность школами, плотностные индикаторы, морфотипы застройки и доступность территории.


In [ ]:
# Обеспеченность школами
from blocksnet.config import service_types_config
from blocksnet.analysis.provision import competitive_provision

school_cfg = service_types_config['school']
demand_norm = int(school_cfg['demand'])
accessibility_norm = int(school_cfg['accessibility'])
print('Норматив: 1 место на', demand_norm, 'жителей; доступность', accessibility_norm, 'минут')

df = buildings_blocks[['population']].join(schools_blocks[['capacity']])
df['capacity'] = df['capacity'].fillna(0).astype(int)
print('Входная таблица спроса и ёмкости:')
display(df.head())

prov_df, links_df = competitive_provision(df, acc_mx, accessibility_norm, demand=demand_norm)
save_df(prov_df, 'provision_schools')
save_df(links_df, 'provision_links')
print('Обеспеченность (средняя):', round(float(prov_df['provision_strong'].mean()), 3))
print('Таблица обеспеченности:')
display(prov_df.head())
print('Связки кварталов и школ:')
display(links_df.head())

In [ ]:
# Карта обеспеченности школами
ax = blocks.plot(figsize=(10, 10), color='#eeeeee', edgecolor='#888888', linewidth=0.3)
blocks.join(prov_df).plot('provision_strong', ax=ax, cmap='RdYlGn', vmin=0, vmax=1,
                         legend=True, edgecolor='#888888', linewidth=0.2)
ax.set_title('Обеспеченность школами (доля спроса, покрытого рядом)')
ax.set_axis_off()
plt.show()

In [ ]:
# Плотностные индикаторы
from blocksnet.analysis.indicators import calculate_density_indicators

# float32 для site_area экономит память при последующем area_accessibility.
buildings_blocks['site_area'] = buildings_blocks.area.astype('float32')
density_df = calculate_density_indicators(buildings_blocks)
density_df.head()

### Карты плотностных показателей

В примере `pipeline.ipynb` плотностные показатели показаны только таблицей. Здесь дополнительно строим карты — так видно пространственную картину:

- **FSI** — где застройка самая интенсивная;
- **GSI** — где земля плотнее всего занята зданиями;
- **MXI** — какие кварталы жилые, а какие нет;
- **L** — где высотная, а где малоэтажная застройка;
- **OSR** — где достаточно свободного пространства;
- **share_living** — компактность жилой застройки.


In [ ]:
# Карты плотностных показателей по кварталам
density_cols = ['fsi', 'gsi', 'mxi', 'l', 'osr', 'share_living']
density_titles = {
    'fsi': 'FSI — плотность застройки, м²/м²',
    'gsi': 'GSI — доля застроенной земли',
    'mxi': 'MXI — доля жилья',
    'l': 'L — средняя этажность',
    'osr': 'OSR — открытые пространства, м²/м²',
    'share_living': 'Жилая площадь на единицу пятна застройки',
}

density_map = blocks.join(density_df)
density_map[density_cols] = density_map[density_cols].replace([float('inf'), -float('inf')], float('nan'))

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, col in zip(axes.ravel(), density_cols):
    if col not in density_map.columns:
        ax.set_axis_off()
        continue
    density_map.plot(column=col, ax=ax, cmap='viridis', legend=True,
                     edgecolor='#888888', linewidth=0.1,
                     missing_kwds={'color': '#eeeeee'})
    ax.set_title(density_titles[col], fontsize=10)
    ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Морфотипы Spacematrix
from blocksnet.analysis.morphotypes import get_spacematrix_morphotypes, get_strelka_morphotypes

# Число кластеров не может быть больше числа застроенных кварталов (важно для теста)
developed = int((density_df['fsi'] > 0).sum())
n_clusters = max(1, min(11, developed))
print('Застроенных кварталов:', developed, '| кластеров Spacematrix:', n_clusters)
spacematrix_df, spacematrix_clusters = get_spacematrix_morphotypes(density_df, n_clusters=n_clusters)

print('Морфотипы по кварталам:')
display(spacematrix_df.head())
print('Портреты кластеров:')
display(spacematrix_clusters.head())

ax = blocks.plot(figsize=(10, 10), color='#eeeeee', edgecolor='#888888', linewidth=0.3)
blocks.join(spacematrix_df[['morphotype']]).plot('morphotype', legend=True, ax=ax,
                                                edgecolor='#888888', linewidth=0.2)
ax.set_title('Морфотипы Spacematrix')
ax.set_axis_off()
plt.show()

In [ ]:
# Морфотипы Strelka
strelka_df = get_strelka_morphotypes(density_df)

print('Морфотипы Strelka по кварталам:')
display(strelka_df.head())

ax = blocks.plot(figsize=(10, 10), color='#eeeeee', edgecolor='#888888', linewidth=0.3)
blocks.join(strelka_df[['morphotype']]).plot('morphotype', legend=True, ax=ax,
                                             edgecolor='#888888', linewidth=0.2)
ax.set_title('Морфотипы Strelka')
ax.set_axis_off()
plt.show()

In [ ]:
# Транспортная доступность территории
from blocksnet.analysis.network import area_accessibility

# area_accessibility взвешивает время по площади квартала. Берём все кварталы
# (а не только застроенные) и float32, чтобы не создавать матрицу float64.
area_blocks = blocks[['geometry']].copy()
area_blocks['site_area'] = area_blocks.area.astype('float32')

gc.collect()
memory_report('перед area_accessibility')
area_acc = area_accessibility(acc_mx, area_blocks)
del area_blocks
gc.collect()

print('Доступность территории по кварталам:')
display(area_acc.head())

ax = blocks.plot(figsize=(10, 10), color='#eeeeee', edgecolor='#888888', linewidth=0.3)
blocks.join(area_acc).plot('area_accessibility', legend=True, ax=ax,
                           edgecolor='#888888', linewidth=0.2)
ax.set_title('Транспортная доступность территории (минуты)')
ax.set_axis_off()
plt.show()

In [ ]:
# Собираем все результаты в одну таблицу и сохраняем
results = blocks.copy()
results = results.join(buildings_blocks[['population']])
results = results.join(schools_blocks[['capacity']])
results = results.join(prov_df[['provision_strong', 'provision_weak']])
results = results.join(area_acc)
results = results.join(spacematrix_df[['morphotype']].rename(columns={'morphotype': 'morphotype_spacematrix'}))
results = results.join(strelka_df[['morphotype']].rename(columns={'morphotype': 'morphotype_strelka'}))
results = results.join(density_df)

save_gdf(results, 'results_blocks')
save_df(density_df, 'density_indicators')
save_df(spacematrix_df, 'morphotypes_spacematrix')
save_df(spacematrix_clusters, 'morphotypes_spacematrix_clusters')
save_df(strelka_df, 'morphotypes_strelka')
save_df(prov_df, 'provision_schools')
save_df(links_df, 'provision_links')
save_df(area_acc, 'area_accessibility')
save_df(results.drop(columns='geometry'), 'results_table')

print('Готово. Столбцов в итоговой таблице:', results.shape[1])
results.head()

### Сводная таблица показателей

Собираем вместе ключевые показатели по всем кварталам и смотрим разброс: среднее, медиану, минимум и максимум. Это позволяет сразу понять типичные значения и выбросы по городу.


In [ ]:
# Сводная статистика по ключевым показателям
summary_cols = [
    'population', 'capacity', 'provision_strong', 'provision_weak',
    'area_accessibility', 'fsi', 'gsi', 'mxi', 'l', 'osr',
    'share_living', 'share_non_living',
]
summary_cols = [c for c in summary_cols if c in results.columns]
results_summary = results[summary_cols].describe().T.round(3)
save_df(results_summary, 'results_summary')

print('Все столбцы итоговой таблицы:')
display(sorted(results.columns))
print('Морфотипы Spacematrix (число кварталов):')
display(results['morphotype_spacematrix'].value_counts(dropna=False))
print('Морфотипы Strelka (число кварталов):')
display(results['morphotype_strelka'].value_counts(dropna=False))
print('Сводная таблица показателей:')
results_summary

## 11. Проверки качества refined

Проверяются размеры таблиц, полнота показателей, диагональ и пропуски матрицы,
суммы населения и ёмкости, геометрия кварталов и отсутствие записи в `original`.
Результат сохраняется в `quality_report.json`.


In [ ]:
# Проверки качества refined.
checks = {}

checks['data_dir_refined'] = DATA_DIR.name == 'refined' and 'original' not in str(DATA_DIR).lower()
checks['blocks_present'] = len(blocks) > 0
checks['all_geometries_valid'] = bool(blocks.geometry.is_valid.all())
checks['unique_block_ids'] = bool(blocks.index.is_unique)
checks['matrix_shape'] = bool(acc_mx.shape == (len(blocks), len(blocks)))
checks['matrix_index'] = bool(acc_mx.index.equals(blocks.index) and acc_mx.columns.equals(blocks.index))
checks['matrix_diagonal_zero'] = bool(np.nanmax(np.abs(acc_mx.values.diagonal())) < 1e-6)
checks['matrix_no_nan'] = bool(np.isfinite(acc_mx.values).all())
checks['population_positive'] = bool(float(buildings_blocks['population'].sum()) > 0)
checks['school_capacity_positive'] = bool(float(schools_blocks['capacity'].sum()) > 0)
checks['results_rows_match'] = bool(len(results) == len(blocks))
checks['results_columns_match'] = bool('population' in results.columns and 'capacity' in results.columns)

required_columns = [
    'population', 'capacity', 'provision_strong', 'provision_weak',
    'area_accessibility', 'fsi', 'gsi', 'mxi', 'l', 'osr',
    'morphotype_spacematrix', 'morphotype_strelka',
]
missing = [c for c in required_columns if c not in results.columns]
checks['all_required_metrics'] = not missing
checks['land_use_present'] = bool('land_use' in results.columns)

original_table = BASE_DIR / 'data_tyumen' / 'original' / 'results_table.csv'
if original_table.exists():
    original_cols = list(pd.read_csv(original_table, nrows=1).columns)
    refined_cols = list(results.drop(columns='geometry').reset_index().columns)
    checks['same_columns_as_original'] = set(original_cols) == set(refined_cols)
else:
    checks['same_columns_as_original'] = None

checks_df = pd.DataFrame([
    {'Проверка': k, 'Результат': ('OK' if v else 'НЕТ') if isinstance(v, bool) else str(v)}
    for k, v in checks.items()
])
display(checks_df)

print('Кварталов:', len(blocks))
print('Население, всего:', int(buildings_blocks['population'].sum()))
print('Школ:', len(schools), '| Ёмкость, мест:', int(schools_blocks['capacity'].sum()))
print('Матрица:', acc_mx.shape)
print('Пропущенные показатели:', missing if missing else 'нет')
print('ИТОГ:', 'все проверки пройдены' if all(v for v in checks.values() if isinstance(v, bool)) else 'есть замечания')

save_json({
    'checks': {k: (bool(v) if isinstance(v, bool) else v) for k, v in checks.items()},
    'blocks': int(len(blocks)),
    'population': int(buildings_blocks['population'].sum()),
    'schools': int(len(schools)),
    'capacity': int(schools_blocks['capacity'].sum()),
    'matrix_shape': [int(acc_mx.shape[0]), int(acc_mx.shape[1])],
    'missing_metrics': missing,
}, 'quality_report')


## 12. Сравнение original и refined

Сравнение читает **только сохранённые** результаты двух версий:
`data_tyumen/original/` и `data_tyumen/refined/`. Оно ничего не пересчитывает
и не изменяет.

Сравниваются:

- число кварталов, средняя, медианная и максимальная площадь;
- распределение населения и доля кварталов без населения;
- число школ и суммарная ёмкость;
- обеспеченность школами и доступность территории;
- корреляция населения с площадью квартала и с жилой площадью;
- доля площади и населения топ-4 и топ-20 кварталов.

> Ёмкость школ в обеих версиях — модельная (250 мест на школу после
> дедупликации в refined). Это не фактическая вместимость школ Тюмени.


In [ ]:
# Сравнительная таблица original vs refined.
ORIGINAL_DIR = BASE_DIR / 'data_tyumen' / 'original'
REFINED_DIR = DATA_DIR

if not (ORIGINAL_DIR / 'results_table.csv').exists():
    raise RuntimeError(
        f'Нет {ORIGINAL_DIR / "results_table.csv"}. Сначала сохраните original-версию.'
    )
if not (REFINED_DIR / 'results_table.csv').exists():
    raise RuntimeError('Нет refined/results_table.csv. Сначала выполните расчёты refined.')

orig_results = pd.read_csv(ORIGINAL_DIR / 'results_table.csv', index_col=0)
ref_results = pd.read_csv(REFINED_DIR / 'results_table.csv', index_col=0)
orig_results.index = orig_results.index.astype(int)
ref_results.index = ref_results.index.astype(int)

orig_blocks = gpd.read_file(ORIGINAL_DIR / 'blocks.geojson')
if 'id' in orig_blocks.columns:
    orig_blocks = orig_blocks.set_index('id')
orig_blocks = orig_blocks.to_crs(utm)
ref_blocks = blocks.copy()
orig_blocks.index = orig_blocks.index.astype(int)
ref_blocks.index = ref_blocks.index.astype(int)

orig_area = orig_blocks.area / 1e6
ref_area = ref_blocks.area / 1e6
orig_pop = orig_results['population']
ref_pop = ref_results['population']


def _top_share(area, pop, n):
    idx = area.nlargest(n).index
    return (float(area.loc[idx].sum() / area.sum() * 100),
            float(pop.loc[idx].sum() / pop.sum() * 100))


def _corr(a, b):
    return float(a.corr(b))


def _delta_pct(old, new):
    return float((new - old) / old * 100) if old not in (0, None) and pd.notna(old) else np.nan


orig_top4_area, orig_top4_pop = _top_share(orig_area, orig_pop, 4)
ref_top4_area, ref_top4_pop = _top_share(ref_area, ref_pop, 4)
orig_top20_area, orig_top20_pop = _top_share(orig_area, orig_pop, 20)
ref_top20_area, ref_top20_pop = _top_share(ref_area, ref_pop, 20)

rows = [
    ('Кварталов, шт.', len(orig_blocks), len(ref_blocks)),
    ('Площадь кварталов, км²', orig_area.sum(), ref_area.sum()),
    ('Средняя площадь, км²', orig_area.mean(), ref_area.mean()),
    ('Медианная площадь, км²', orig_area.median(), ref_area.median()),
    ('Максимальная площадь, км²', orig_area.max(), ref_area.max()),
    ('Кварталов > 10 км², шт.', int((orig_area > 10).sum()), int((ref_area > 10).sum())),
    ('Население, чел.', orig_pop.sum(), ref_pop.sum()),
    ('Кварталов без населения, %', (orig_pop == 0).mean() * 100, (ref_pop == 0).mean() * 100),
    ('Школ, шт.', len(gpd.read_file(ORIGINAL_DIR / 'schools.geojson')),
     len(gpd.read_file(REFINED_DIR / 'schools.geojson'))),
    ('Ёмкость школ, мест', orig_results['capacity'].sum(), ref_results['capacity'].sum()),
    ('Средняя обеспеченность', orig_results['provision_strong'].mean(),
     ref_results['provision_strong'].mean()),
    ('Средняя доступность территории, мин', orig_results['area_accessibility'].mean(),
     ref_results['area_accessibility'].mean()),
    ('Корреляция населения и площади', _corr(orig_pop, orig_area), _corr(ref_pop, ref_area)),
    ('Корреляция населения и жилой площади', _corr(orig_pop, orig_results['living_area']),
     _corr(ref_pop, ref_results['living_area'])),
    ('Топ-4: доля площади, %', orig_top4_area, ref_top4_area),
    ('Топ-4: доля населения, %', orig_top4_pop, ref_top4_pop),
    ('Топ-20: доля площади, %', orig_top20_area, ref_top20_area),
    ('Топ-20: доля населения, %', orig_top20_pop, ref_top20_pop),
]

comparison = pd.DataFrame(rows, columns=['Показатель', 'Original', 'Refined'])
comparison['Δ'] = comparison['Refined'] - comparison['Original']
comparison['Δ, %'] = comparison.apply(
    lambda r: _delta_pct(r['Original'], r['Refined']), axis=1
)
comparison = comparison.set_index('Показатель')
for c in ['Original', 'Refined', 'Δ']:
    comparison[c] = pd.to_numeric(comparison[c], errors='coerce').round(3)
comparison['Δ, %'] = pd.to_numeric(comparison['Δ, %'], errors='coerce').round(1)

display(comparison)
save_df(comparison, 'comparison')
save_json({
    'original_dir': str(ORIGINAL_DIR),
    'refined_dir': str(REFINED_DIR),
    'original_blocks': int(len(orig_blocks)),
    'refined_blocks': int(len(ref_blocks)),
    'original_population': int(orig_pop.sum()),
    'refined_population': int(ref_pop.sum()),
    'original_schools': int(len(gpd.read_file(ORIGINAL_DIR / 'schools.geojson'))),
    'refined_schools': int(len(gpd.read_file(REFINED_DIR / 'schools.geojson'))),
    'original_capacity': int(orig_results['capacity'].sum()),
    'refined_capacity': int(ref_results['capacity'].sum()),
    'original_provision_mean': float(orig_results['provision_strong'].mean()),
    'refined_provision_mean': float(ref_results['provision_strong'].mean()),
    'original_area_accessibility_mean': float(orig_results['area_accessibility'].mean()),
    'refined_area_accessibility_mean': float(ref_results['area_accessibility'].mean()),
    'note': 'school capacity is modeled (250 per deduplicated school in refined)',
}, 'comparison')


In [ ]:
# Пространственное сравнение: original и refined рядом.
metric_panels = [
    ('population', 'Население, чел.', 'Reds'),
    ('provision_strong', 'Обеспеченность школами', 'viridis'),
    ('area_accessibility', 'Доступность территории, мин', 'RdYlGn_r'),
]

fig, axes = plt.subplots(len(metric_panels), 2, figsize=(18, 20))
for row, (column, title, cmap) in enumerate(metric_panels):
    # Matplotlib names are case-sensitive.
    if str(cmap).lower() == 'viridis':
        cmap = 'viridis'
    if column not in orig_results.columns or column not in ref_results.columns:
        for ax in axes[row]:
            ax.set_axis_off()
            ax.set_title(f'{title}: показатель отсутствует')
        continue
    orig_map = orig_blocks.join(orig_results[[column]])
    ref_map = ref_blocks.join(ref_results[[column]])
    vals = np.r_[orig_map[column].values, ref_map[column].values]
    vmin = float(np.nanpercentile(vals, 1))
    vmax = float(np.nanpercentile(vals, 99))
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin, vmax = None, None
    for ax, gdf, label in [
        (axes[row, 0], orig_map, f'Original: {len(orig_map)} кварталов'),
        (axes[row, 1], ref_map, f'Refined: {len(ref_map)} кварталов'),
    ]:
        gdf.plot(column=column, ax=ax, cmap=cmap, vmin=vmin, vmax=vmax,
                 legend=True, edgecolor='#888888', linewidth=0.1,
                 missing_kwds={'color': '#eeeeee'})
        ax.set_title(f'{title}\n{label}', fontsize=10)
        ax.set_axis_off()

plt.suptitle('Пространственное распределение показателей: original vs refined')
plt.tight_layout()
plt.show()


## 13. Повторный запуск refined

1. Ноутбук не скачивает данные: локальный PBF уже лежит в
   `examples/data_tyumen/refined/tyumen_refined.osm.pbf`.
2. Все результаты пишутся только в `examples/data_tyumen/refined/`.
   Папка `original/` не изменяется.
3. Матрица доступности считается порциями в
   `refined/accessibility_matrix_parts/`. После сбоя запустите ячейку матрицы
   снова: готовые порции переиспользуются.
4. Если изменить `SPLIT_MAX_AREA_KM2`, `MIN_BLOCK_AREA_M2` или
   `SPLIT_LARGE_BLOCKS`, удалите `blocks_initial.geojson`, `blocks_refined.geojson`,
   `split_meta.json` и производные файлы. Ноутбук также проверяет подпись
   геометрии и сам не использует старую матрицу при изменении кварталов.
5. Чтобы пересчитать всё «с нуля», удалите производные файлы в `refined/`,
   но сохраните входные: `tyumen_refined.osm.pbf`, `boundaries.geojson`,
   `functional_zones.geojson`, `graph.pkl`, `railways.geojson`, `water.geojson`,
   `buildings_raw.geojson`, `schools_raw.geojson`.
6. Для сравнения нужны обе папки: `original/` и `refined/`.

### Файлы refined

- `blocks_initial.geojson` — базовая нарезка до split
- `blocks_refined.geojson`, `blocks.geojson` — итоговая нарезка
- `split_meta.json` — параметры split и подписи геометрии
- `blocks_quality.csv`, `blocks_quality.json` — проверки геометрии
- `blocks_land_use.geojson` — землепользование
- `buildings.geojson`, `buildings_blocks.geojson` — здания и агрегаты
- `schools_raw.geojson`, `schools.geojson`, `schools_blocks.geojson` — школы
- `accessibility_matrix.parquet` / `.csv`, `accessibility_matrix_parts/`,
  `matrix_meta.json` — матрица
- `provision_schools.csv`, `provision_links.csv` — обеспеченность
- `density_indicators.csv` — плотность
- `morphotypes_spacematrix.csv`, `morphotypes_spacematrix_clusters.csv`,
  `morphotypes_strelka.csv` — морфотипы
- `area_accessibility.csv` — доступность территории
- `results_blocks.geojson`, `results_table.csv`, `results_summary.csv` — итоги
- `quality_report.json` — проверки refined
- `comparison.csv`, `comparison.json` — сравнение с original
